In [ ]:
# ========================== MOUNT GOOGLE DRIVE ==========================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 34.6 MB/s eta 0:00:00


In [ ]:
!pip install supervision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.4/212.4 kB 8.9 MB/s eta 0:00:00


In [ ]:
# ========================== IMPORTS ==========================
import os
import cv2
import torch
import numpy as np
from pathlib import Path
import json
from tqdm import tqdm
from ultralytics import SAM
from transformers import AutoImageProcessor, AutoModel

# ========================== LOAD MODELS ==========================
print("Đang load MobileSAM + DINOv2-small + float16...")
sam = SAM("mobile_sam.pt")

processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
dinov2 = AutoModel.from_pretrained("facebook/dinov2-small").cuda().eval()

# ========================== EMBEDDING BATCH ==========================
@torch.inference_mode()
def batch_embed(crops):
    if len(crops) == 0:
        return torch.empty((0, 384), device="cuda")
    inputs = processor(images=crops, return_tensors="pt").to("cuda")
    with torch.autocast("cuda", dtype=torch.float16):
        out = dinov2(**inputs)
    patches = out.last_hidden_state[:, 1:]
    topk = patches.topk(16, dim=1).values.mean(dim=1)
    return torch.nn.functional.normalize(topk, dim=1)

# ========================== REFERENCE TEMPLATE ==========================
def get_template(ref_paths, sam_model):
    imgs = []
    for p in ref_paths:
        img_bgr = cv2.imread(str(p))
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        crop = img_rgb

        results = sam_model(img_rgb, imgsz=768, device="cuda", verbose=False)[0]

        if results.masks is not None and len(results.masks) > 0:
            masks = results.masks.data.cpu().numpy()

            scores = None
            if results.boxes is not None and results.boxes.conf is not None:
                scores = results.boxes.conf.detach().cpu().numpy()
            elif hasattr(results.masks, "scores") and results.masks.scores is not None:
                scores = results.masks.scores.detach().cpu().numpy()

            if scores is not None and len(scores) == len(masks):
                best_idx = int(np.argmax(scores))
            else:
                areas = masks.sum(axis=(1, 2))
                best_idx = int(np.argmax(areas))

            best_mask = masks[best_idx]
            pos = np.where(best_mask)

            if pos[0].size > 100:
                y1, y2 = pos[0].min(), pos[0].max()
                x1, x2 = pos[1].min(), pos[1].max()

                h, w = img_rgb.shape[:2]
                pad_y = max(1, int((y2 - y1 + 1) * 0.05))
                pad_x = max(1, int((x2 - x1 + 1) * 0.05))
                y1 = max(0, y1 - pad_y)
                y2 = min(h - 1, y2 + pad_y)
                x1 = max(0, x1 - pad_x)
                x2 = min(w - 1, x2 + pad_x)

                crop = img_rgb[y1:y2+1, x1:x2+1]

        if crop.size == 0:
            crop = img_rgb
        if crop is img_rgb:
            h, w = img_rgb.shape[:2]
            margin_h, margin_w = max(1, int(h * 0.2)), max(1, int(w * 0.2))
            crop = img_rgb[margin_h:h-margin_h, margin_w:w-margin_w]
            if crop.size == 0:
                crop = img_rgb

        crop_resized = cv2.resize(crop, (224, 224))
        imgs.append(crop_resized)

    if len(imgs) == 0:
        raise ValueError("Không tạo được crop nào từ ảnh tham chiếu.")

    embs = batch_embed(imgs)
    template = embs.mean(dim=0)
    return torch.nn.functional.normalize(template, dim=0)

# ========================== HELPER FUNCTIONS ==========================
def bbox_center(bbox):
    """Lấy tâm của bbox [x1, y1, x2, y2]"""
    return ((bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2)

def bbox_distance(bbox1, bbox2):
    """Tính khoảng cách giữa tâm 2 bbox"""
    c1 = bbox_center(bbox1)
    c2 = bbox_center(bbox2)
    return np.sqrt((c1[0] - c2[0])**2 + (c1[1] - c2[1])**2)

def bbox_iou(bbox1, bbox2):
    """Tính IoU giữa 2 bbox"""
    x1 = max(bbox1[0], bbox2[0])
    y1 = max(bbox1[1], bbox2[1])
    x2 = min(bbox1[2], bbox2[2])
    y2 = min(bbox1[3], bbox2[3])

    if x2 <= x1 or y2 <= y1:
        return 0.0

    inter = (x2 - x1) * (y2 - y1)
    area1 = (bbox1[2] - bbox1[0]) * (bbox1[3] - bbox1[1])
    area2 = (bbox2[2] - bbox2[0]) * (bbox2[3] - bbox2[1])
    union = area1 + area2 - inter

    return inter / union if union > 0 else 0.0

# ========================== MAIN PROCESS ==========================
@torch.inference_mode()
def process_video(folder: Path):
    video_path = folder / "drone_video_test_1.mp4"
    refs = sorted((folder / "object_images").glob("*.jpg"))
    video_id = folder.name

    template = get_template(refs, sam)

    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    detections = []
    skip_counter = 0

    # Tracking state
    last_bbox = None  # Bbox của frame trước
    frames_since_last_det = 0  # Số frame từ lần detect cuối
    max_frames_lost = 30  # Reset tracking sau 30 frames không detect

    pbar = tqdm(total=total_frames, desc=video_id, leave=False)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx = int(pbar.n)
        img_h, img_w = frame.shape[:2]

        # SKIP FRAME: chỉ detect mỗi 2 frame
        should_detect = (skip_counter % 2 == 0)

        if should_detect:
            results = sam(frame, imgsz=768, device="cuda", verbose=False)[0]

            if results.masks is None:
                frames_since_last_det += 1
                if frames_since_last_det > max_frames_lost:
                    last_bbox = None
                pbar.update(1)
                skip_counter += 1
                continue

            masks = results.masks.data.cpu().numpy()

            # Lọc masks theo pixel
            min_pixels = 70
            max_pixels = 4000
            valid_masks = []
            valid_areas = []

            for mask in masks:
                pos = np.where(mask)
                num_pixels = len(pos[0])
                if min_pixels <= num_pixels <= max_pixels:
                    valid_masks.append(mask)
                    valid_areas.append(num_pixels)

            # Lấy top 24 lớn nhất
            if len(valid_masks) > 24:
                topk_idx = np.argsort(valid_areas)[-24:]
                masks = [valid_masks[i] for i in topk_idx]
            else:
                masks = valid_masks

            # Lọc bbox quá lớn
            max_bbox_width = img_w * 0.2
            max_bbox_height = img_h * 0.2
            max_bbox_area = (img_w * img_h) * 0.05

            size_filtered_masks = []
            for mask in masks:
                pos = np.where(mask)
                if pos[0].size == 0:
                    continue
                y1, y2 = pos[0].min(), pos[0].max()
                x1, x2 = pos[1].min(), pos[1].max()

                bbox_w = x2 - x1
                bbox_h = y2 - y1
                bbox_area = bbox_w * bbox_h

                if bbox_w > max_bbox_width or bbox_h > max_bbox_height or bbox_area > max_bbox_area:
                    continue

                size_filtered_masks.append(mask)

            masks = size_filtered_masks

            # Crop và lấy bbox
            crops_rgb = []
            bboxes = []
            for mask in masks:
                pos = np.where(mask)
                if pos[0].size == 0:
                    continue
                y1, y2 = pos[0].min(), pos[0].max()
                x1, x2 = pos[1].min(), pos[1].max()

                crop = frame[y1:y2+1, x1:x2+1]
                crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                crops_rgb.append(cv2.resize(crop_rgb, (224, 224)))
                bboxes.append([x1, y1, x2, y2])

            # So khớp với template
            if crops_rgb:
                embs = batch_embed(crops_rgb)
                sims = torch.nn.functional.cosine_similarity(embs, template, dim=1)

                # Lấy tất cả candidates có similarity > threshold
                sim_threshold = 0.82
                candidates = []
                for idx in range(len(sims)):
                    if sims[idx].item() > sim_threshold:
                        candidates.append({
                            'idx': idx,
                            'sim': sims[idx].item(),
                            'bbox': bboxes[idx]
                        })

                if candidates:
                    best_candidate = None

                    if last_bbox is None:
                        # Chưa có tracking -> chọn similarity cao nhất
                        best_candidate = max(candidates, key=lambda x: x['sim'])
                    else:
                        # Đã có tracking -> ưu tiên gần vị trí cũ
                        # Tính max velocity dựa trên số frame đã qua
                        max_velocity = 50 * (frames_since_last_det + 1)  # pixels per frame
                        max_distance = min(max_velocity, 200)  # cap at 200 pixels

                        # Lọc candidates trong vùng hợp lý
                        nearby_candidates = []
                        for cand in candidates:
                            dist = bbox_distance(last_bbox, cand['bbox'])
                            if dist <= max_distance:
                                # Tính combined score: similarity + proximity
                                proximity_score = 1 - (dist / max_distance)
                                combined_score = 0.6 * cand['sim'] + 0.4 * proximity_score
                                cand['combined_score'] = combined_score
                                cand['distance'] = dist
                                nearby_candidates.append(cand)

                        if nearby_candidates:
                            # Chọn candidate có combined score cao nhất
                            best_candidate = max(nearby_candidates, key=lambda x: x['combined_score'])
                        else:
                            # Không có candidate nào gần -> có thể object mới xuất hiện
                            # Chỉ chấp nhận nếu similarity rất cao
                            high_sim_candidates = [c for c in candidates if c['sim'] > 0.85]
                            if high_sim_candidates:
                                best_candidate = max(high_sim_candidates, key=lambda x: x['sim'])
                                # Reset tracking vì đây là object mới
                                last_bbox = None

                    if best_candidate:
                        x1, y1, x2, y2 = best_candidate['bbox']
                        detections.append({
                            "frame": frame_idx,
                            "x1": int(x1),
                            "y1": int(y1),
                            "x2": int(x2),
                            "y2": int(y2)
                        })
                        last_bbox = best_candidate['bbox']
                        frames_since_last_det = 0
                    else:
                        frames_since_last_det += 1
                else:
                    frames_since_last_det += 1
            else:
                frames_since_last_det += 1

            # Reset tracking nếu mất quá lâu
            if frames_since_last_det > max_frames_lost:
                last_bbox = None

        skip_counter += 1
        pbar.update(1)

    cap.release()
    pbar.close()

    # INTERPOLATION với kiểm tra chặt chẽ
    if len(detections) > 1:
        full_dets = []
        max_gap = 5  # Giảm xuống 5 frames
        max_interp_distance = 80  # Khoảng cách tối đa để interpolate

        sorted_dets = sorted(detections, key=lambda x: x["frame"])

        for i in range(len(sorted_dets)):
            det = sorted_dets[i]
            full_dets.append(det)

            if i < len(sorted_dets) - 1:
                next_det = sorted_dets[i + 1]
                gap = next_det["frame"] - det["frame"]

                # Tính khoảng cách
                bbox1 = [det["x1"], det["y1"], det["x2"], det["y2"]]
                bbox2 = [next_det["x1"], next_det["y1"], next_det["x2"], next_det["y2"]]
                distance = bbox_distance(bbox1, bbox2)

                # Chỉ interpolate nếu gap nhỏ VÀ khoảng cách hợp lý
                if 1 < gap <= max_gap and distance <= max_interp_distance:
                    for f in range(det["frame"] + 1, next_det["frame"]):
                        alpha = (f - det["frame"]) / gap
                        bbox = {
                            "frame": f,
                            "x1": int(det["x1"] * (1-alpha) + next_det["x1"] * alpha),
                            "y1": int(det["y1"] * (1-alpha) + next_det["y1"] * alpha),
                            "x2": int(det["x2"] * (1-alpha) + next_det["x2"] * alpha),
                            "y2": int(det["y2"] * (1-alpha) + next_det["y2"] * alpha),
                        }
                        full_dets.append(bbox)

        detections = sorted(full_dets, key=lambda x: x["frame"])

    # Format output
    if len(detections) > 0:
        final_output_detections = [{"bboxes": detections}]
    else:
        final_output_detections = []

    return {"video_id": video_id, "detections": final_output_detections}

# ========================== MAIN ==========================
if __name__ == "__main__":
    DATA_DIR = "/content/drive/MyDrive/ZaloAI/public_test/samples"
    OUTPUT_FILE = "/content/submission.json"

    results = []
    data_path = Path(DATA_DIR)

    if not data_path.exists():
        print(f"Lỗi: Không tìm thấy thư mục {DATA_DIR}")
    else:
        folders = sorted([f for f in data_path.iterdir() if (f / "drone_video_test_1.mp4").exists()])
        print(f"Tìm thấy {len(folders)} video để xử lý")

        for folder in folders:
            print(f"Processing {folder.name} ...")
            res = process_video(folder)
            results.append(res)

        json.dump(results, open(OUTPUT_FILE, "w"), indent=2)
        print(f"\nHoàn tất! Output: {OUTPUT_FILE}")

        import shutil
        drive_output = f"{DATA_DIR}/submission.json"
        shutil.copy(OUTPUT_FILE, drive_output)
        print(f"Đã copy kết quả vào: {drive_output}")

Đang load MobileSAM + DINOv2-small + float16...
Tìm thấy 1 video để xử lý
Processing BlackBox_0 ...




BlackBox_0:   0%|          | 0/440 [00:00<?, ?it/s]

BlackBox_0:   0%|          | 1/440 [00:00<05:52,  1.24it/s]

BlackBox_0:   1%|          | 3/440 [00:01<03:34,  2.03it/s]

BlackBox_0:   1%|          | 5/440 [00:02<03:10,  2.28it/s]

BlackBox_0:   2%|▏         | 7/440 [00:03<02:59,  2.41it/s]

BlackBox_0:   2%|▏         | 9/440 [00:03<02:54,  2.47it/s]

BlackBox_0:   2%|▎         | 11/440 [00:04<02:51,  2.50it/s]

BlackBox_0:   3%|▎         | 13/440 [00:05<02:48,  2.53it/s]

BlackBox_0:   3%|▎         | 15/440 [00:06<02:46,  2.55it/s]

BlackBox_0:   4%|▍         | 17/440 [00:06<02:44,  2.57it/s]

BlackBox_0:   4%|▍         | 19/440 [00:07<02:42,  2.59it/s]

BlackBox_0:   5%|▍         | 21/440 [00:08<02:42,  2.58it/s]

BlackBox_0:   5%|▌         | 23/440 [00:09<02:43,  2.56it/s]

KeyboardInterrupt: 

In [ ]:
# ========================== IMPORTS ==========================
import os
import gc
import cv2
import torch
import numpy as np
from pathlib import Path
import json
from tqdm import tqdm
from ultralytics.models.fastsam import FastSAMPredictor
from transformers import AutoImageProcessor, AutoModel

# ========================== LOAD MODELS ==========================
print("Đang load FastSAM + DINOv2-small + float16...")
overrides = dict(conf=0.1, task="segment", mode="predict", model="FastSAM-s.pt", imgsz=768)
sam = FastSAMPredictor(overrides=overrides)

processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
dinov2 = AutoModel.from_pretrained("facebook/dinov2-small").cuda().half().eval()  # Thêm .half()

# ========================== EMBEDDING BATCH ==========================
@torch.inference_mode()
def batch_embed(crops, batch_size=8):  # Thêm batch_size để xử lý từng nhóm nhỏ
    if len(crops) == 0:
        return torch.empty((0, 384), device="cuda")

    all_embs = []
    for i in range(0, len(crops), batch_size):
        batch_crops = crops[i:i+batch_size]
        inputs = processor(images=batch_crops, return_tensors="pt").to("cuda")

        with torch.autocast("cuda", dtype=torch.float16):
            out = dinov2(**inputs)

        patches = out.last_hidden_state[:, 1:]
        topk = patches.topk(16, dim=1).values.mean(dim=1)
        embs = torch.nn.functional.normalize(topk, dim=1)

        all_embs.append(embs.cpu())  # Chuyển về CPU ngay

        # Xóa bộ nhớ
        del inputs, out, patches, topk

    result = torch.cat(all_embs, dim=0).cuda()
    del all_embs
    torch.cuda.empty_cache()

    return result

# ========================== REFERENCE TEMPLATE ==========================
def get_template(ref_paths, sam_model):
    imgs = []
    for p in ref_paths:
        img_bgr = cv2.imread(str(p))
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        crop = img_rgb

        results = sam_model(img_rgb)[0]

        if results.masks is not None and len(results.masks) > 0:
            masks = results.masks.data.cpu().numpy()

            scores = None
            if results.boxes is not None and results.boxes.conf is not None:
                scores = results.boxes.conf.detach().cpu().numpy()
            elif hasattr(results.masks, "scores") and results.masks.scores is not None:
                scores = results.masks.scores.detach().cpu().numpy()

            if scores is not None and len(scores) == len(masks):
                best_idx = int(np.argmax(scores))
            else:
                areas = masks.sum(axis=(1, 2))
                best_idx = int(np.argmax(areas))

            best_mask = masks[best_idx]
            pos = np.where(best_mask)

            if pos[0].size > 100:
                y1, y2 = pos[0].min(), pos[0].max()
                x1, x2 = pos[1].min(), pos[1].max()

                h, w = img_rgb.shape[:2]
                pad_y = max(1, int((y2 - y1 + 1) * 0.05))
                pad_x = max(1, int((x2 - x1 + 1) * 0.05))
                y1 = max(0, y1 - pad_y)
                y2 = min(h - 1, y2 + pad_y)
                x1 = max(0, x1 - pad_x)
                x2 = min(w - 1, x2 + pad_x)

                crop = img_rgb[y1:y2+1, x1:x2+1]

        if crop.size == 0:
            crop = img_rgb
        if crop is img_rgb:
            h, w = img_rgb.shape[:2]
            margin_h, margin_w = max(1, int(h * 0.2)), max(1, int(w * 0.2))
            crop = img_rgb[margin_h:h-margin_h, margin_w:w-margin_w]
            if crop.size == 0:
                crop = img_rgb

        crop_resized = cv2.resize(crop, (224, 224))
        imgs.append(crop_resized)

        # Xóa bộ nhớ
        del img_bgr, img_rgb, crop

    if len(imgs) == 0:
        raise ValueError("Không tạo được crop nào từ ảnh tham chiếu.")

    embs = batch_embed(imgs)
    template = embs.mean(dim=0)
    result = torch.nn.functional.normalize(template, dim=0)

    del embs, template
    torch.cuda.empty_cache()

    return result

# ========================== HELPER FUNCTIONS ==========================
def bbox_center(bbox):
    return ((bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2)

def bbox_distance(bbox1, bbox2):
    c1 = bbox_center(bbox1)
    c2 = bbox_center(bbox2)
    return np.sqrt((c1[0] - c2[0])**2 + (c1[1] - c2[1])**2)

def bbox_iou(bbox1, bbox2):
    x1 = max(bbox1[0], bbox2[0])
    y1 = max(bbox1[1], bbox2[1])
    x2 = min(bbox1[2], bbox2[2])
    y2 = min(bbox1[3], bbox2[3])

    if x2 <= x1 or y2 <= y1:
        return 0.0

    inter = (x2 - x1) * (y2 - y1)
    area1 = (bbox1[2] - bbox1[0]) * (bbox1[3] - bbox1[1])
    area2 = (bbox2[2] - bbox2[0]) * (bbox2[3] - bbox2[1])
    union = area1 + area2 - inter

    return inter / union if union > 0 else 0.0

# ========================== MAIN PROCESS ==========================
@torch.inference_mode()
def process_video(folder: Path):
    video_path = folder / "drone_video_test_1.mp4"
    refs = sorted((folder / "object_images").glob("*.jpg"))
    video_id = folder.name

    template = get_template(refs, sam)

    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    detections = []
    skip_counter = 0

    last_bbox = None
    frames_since_last_det = 0
    max_frames_lost = 30

    pbar = tqdm(total=total_frames, desc=video_id, leave=False)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx = int(pbar.n)
        img_h, img_w = frame.shape[:2]

        should_detect = (skip_counter % 2 == 0)

        if should_detect:
            # Giải phóng cache trước mỗi lần inference
            torch.cuda.empty_cache()

            results = sam(frame)[0]

            if results.masks is None:
                frames_since_last_det += 1
                if frames_since_last_det > max_frames_lost:
                    last_bbox = None
                pbar.update(1)
                skip_counter += 1
                continue

            # Chuyển masks về numpy ngay và xóa tensor
            masks = results.masks.data.cpu().numpy()
            del results
            torch.cuda.empty_cache()

            min_pixels = 70
            max_pixels = 4000
            valid_masks = []
            valid_areas = []

            for mask in masks:
                pos = np.where(mask)
                num_pixels = len(pos[0])
                if min_pixels <= num_pixels <= max_pixels:
                    valid_masks.append(mask)
                    valid_areas.append(num_pixels)

            if len(valid_masks) > 24:
                topk_idx = np.argsort(valid_areas)[-24:]
                masks = [valid_masks[i] for i in topk_idx]
            else:
                masks = valid_masks

            # Xóa bộ nhớ tạm
            del valid_masks, valid_areas

            max_bbox_width = img_w * 0.2
            max_bbox_height = img_h * 0.2
            max_bbox_area = (img_w * img_h) * 0.05

            size_filtered_masks = []
            for mask in masks:
                pos = np.where(mask)
                if pos[0].size == 0:
                    continue
                y1, y2 = pos[0].min(), pos[0].max()
                x1, x2 = pos[1].min(), pos[1].max()

                bbox_w = x2 - x1
                bbox_h = y2 - y1
                bbox_area = bbox_w * bbox_h

                if bbox_w > max_bbox_width or bbox_h > max_bbox_height or bbox_area > max_bbox_area:
                    continue

                size_filtered_masks.append(mask)

            masks = size_filtered_masks
            del size_filtered_masks

            crops_rgb = []
            bboxes = []
            for mask in masks:
                pos = np.where(mask)
                if pos[0].size == 0:
                    continue
                y1, y2 = pos[0].min(), pos[0].max()
                x1, x2 = pos[1].min(), pos[1].max()

                crop = frame[y1:y2+1, x1:x2+1]
                crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                crops_rgb.append(cv2.resize(crop_rgb, (224, 224)))
                bboxes.append([x1, y1, x2, y2])

            # Xóa masks sau khi dùng xong
            del masks

            if crops_rgb:
                embs = batch_embed(crops_rgb)
                sims = torch.nn.functional.cosine_similarity(embs, template, dim=1)

                # Chuyển similarity về CPU để xử lý
                sims_cpu = sims.cpu().numpy()
                del embs, sims
                torch.cuda.empty_cache()

                sim_threshold = 0.82
                candidates = []
                for idx in range(len(sims_cpu)):
                    if sims_cpu[idx] > sim_threshold:
                        candidates.append({
                            'idx': idx,
                            'sim': float(sims_cpu[idx]),
                            'bbox': bboxes[idx]
                        })

                del sims_cpu, crops_rgb, bboxes

                if candidates:
                    best_candidate = None

                    if last_bbox is None:
                        best_candidate = max(candidates, key=lambda x: x['sim'])
                    else:
                        max_velocity = 50 * (frames_since_last_det + 1)
                        max_distance = min(max_velocity, 200)

                        nearby_candidates = []
                        for cand in candidates:
                            dist = bbox_distance(last_bbox, cand['bbox'])
                            if dist <= max_distance:
                                proximity_score = 1 - (dist / max_distance)
                                combined_score = 0.6 * cand['sim'] + 0.4 * proximity_score
                                cand['combined_score'] = combined_score
                                cand['distance'] = dist
                                nearby_candidates.append(cand)

                        if nearby_candidates:
                            best_candidate = max(nearby_candidates, key=lambda x: x['combined_score'])
                        else:
                            high_sim_candidates = [c for c in candidates if c['sim'] > 0.85]
                            if high_sim_candidates:
                                best_candidate = max(high_sim_candidates, key=lambda x: x['sim'])
                                last_bbox = None

                    if best_candidate:
                        x1, y1, x2, y2 = best_candidate['bbox']
                        detections.append({
                            "frame": frame_idx,
                            "x1": int(x1),
                            "y1": int(y1),
                            "x2": int(x2),
                            "y2": int(y2)
                        })
                        last_bbox = best_candidate['bbox']
                        frames_since_last_det = 0
                    else:
                        frames_since_last_det += 1
                else:
                    frames_since_last_det += 1
            else:
                frames_since_last_det += 1

            if frames_since_last_det > max_frames_lost:
                last_bbox = None

        skip_counter += 1
        pbar.update(1)

        # Giải phóng bộ nhớ định kỳ mỗi 50 frames
        if frame_idx % 50 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    cap.release()
    pbar.close()

    # Giải phóng bộ nhớ sau khi xử lý xong video
    gc.collect()
    torch.cuda.empty_cache()

    # INTERPOLATION
    if len(detections) > 1:
        full_dets = []
        max_gap = 5
        max_interp_distance = 80

        sorted_dets = sorted(detections, key=lambda x: x["frame"])

        for i in range(len(sorted_dets)):
            det = sorted_dets[i]
            full_dets.append(det)

            if i < len(sorted_dets) - 1:
                next_det = sorted_dets[i + 1]
                gap = next_det["frame"] - det["frame"]

                bbox1 = [det["x1"], det["y1"], det["x2"], det["y2"]]
                bbox2 = [next_det["x1"], next_det["y1"], next_det["x2"], next_det["y2"]]
                distance = bbox_distance(bbox1, bbox2)

                if 1 < gap <= max_gap and distance <= max_interp_distance:
                    for f in range(det["frame"] + 1, next_det["frame"]):
                        alpha = (f - det["frame"]) / gap
                        bbox = {
                            "frame": f,
                            "x1": int(det["x1"] * (1-alpha) + next_det["x1"] * alpha),
                            "y1": int(det["y1"] * (1-alpha) + next_det["y1"] * alpha),
                            "x2": int(det["x2"] * (1-alpha) + next_det["x2"] * alpha),
                            "y2": int(det["y2"] * (1-alpha) + next_det["y2"] * alpha),
                        }
                        full_dets.append(bbox)

        detections = sorted(full_dets, key=lambda x: x["frame"])

    if len(detections) > 0:
        final_output_detections = [{"bboxes": detections}]
    else:
        final_output_detections = []

    return {"video_id": video_id, "detections": final_output_detections}

# ========================== MAIN ==========================
if __name__ == "__main__":
    DATA_DIR = "/content/drive/MyDrive/ZaloAI/public_test/samples"
    OUTPUT_FILE = "/content/submission.json"

    results = []
    data_path = Path(DATA_DIR)

    if not data_path.exists():
        print(f"Lỗi: Không tìm thấy thư mục {DATA_DIR}")
    else:
        folders = sorted([f for f in data_path.iterdir() if (f / "drone_video_test_1.mp4").exists()])
        print(f"Tìm thấy {len(folders)} video để xử lý")

        for folder in folders:
            print(f"Processing {folder.name} ...")
            res = process_video(folder)
            results.append(res)

            # Giải phóng bộ nhớ sau mỗi video
            gc.collect()
            torch.cuda.empty_cache()

        json.dump(results, open(OUTPUT_FILE, "w"), indent=2)
        print(f"\nHoàn tất! Output: {OUTPUT_FILE}")

        import shutil
        drive_output = f"{DATA_DIR}/submission.json"
        shutil.copy(OUTPUT_FILE, drive_output)
        print(f"Đã copy kết quả vào: {drive_output}")

ModuleNotFoundError: No module named 'ultralytics'

In [ ]:
# ========================== IMPORTS ==========================
import os
import gc
import cv2
import torch
import numpy as np
from pathlib import Path
import json
from tqdm import tqdm
from ultralytics.models.fastsam import FastSAMPredictor
from ultralytics import SAM
from transformers import AutoImageProcessor, AutoModel


# ========================== LOAD MODELS ==========================
print("Đang load FastSAM + DINOv2-small + float16...")

# Giữ nguyên Predictor
overrides = dict(
    conf=0.25,
    task="segment",
    mode="predict",
    model="FastSAM-s.pt",
    imgsz=640,
    retina_masks=False
)

sam_template = SAM("mobile_sam.pt")
sam = FastSAMPredictor(overrides=overrides)

processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
dinov2 = AutoModel.from_pretrained("facebook/dinov2-small").cuda().half().eval()

# ========================== EMBEDDING BATCH ==========================
@torch.inference_mode()
def batch_embed(crops, batch_size=4):
    if len(crops) == 0:
        return torch.empty((0, 384), device="cuda")

    all_embs = []
    for i in range(0, len(crops), batch_size):
        batch_crops = crops[i:i+batch_size]
        inputs = processor(images=batch_crops, return_tensors="pt").to("cuda")

        with torch.autocast("cuda", dtype=torch.float16):
            out = dinov2(**inputs)

        patches = out.last_hidden_state[:, 1:]
        topk = patches.topk(16, dim=1).values.mean(dim=1)
        embs = torch.nn.functional.normalize(topk, dim=1)

        all_embs.append(embs.cpu())

        del inputs, out, patches, topk, embs

    result = torch.cat(all_embs, dim=0).cuda()
    del all_embs
    torch.cuda.empty_cache()

    return result

# ========================== REFERENCE TEMPLATE ==========================
def get_template(ref_paths, sam_model):
    imgs = []
    for p in ref_paths:
        img_bgr = cv2.imread(str(p))
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        crop = img_rgb

        results = sam_model(img_rgb, imgsz=768, device="cuda", verbose=False)[0]

        if results.masks is not None and len(results.masks) > 0:
            masks = results.masks.data.cpu().numpy()

            scores = None
            if results.boxes is not None and results.boxes.conf is not None:
                scores = results.boxes.conf.detach().cpu().numpy()
            elif hasattr(results.masks, "scores") and results.masks.scores is not None:
                scores = results.masks.scores.detach().cpu().numpy()

            if scores is not None and len(scores) == len(masks):
                best_idx = int(np.argmax(scores))
            else:
                areas = masks.sum(axis=(1, 2))
                best_idx = int(np.argmax(areas))

            best_mask = masks[best_idx]
            pos = np.where(best_mask)

            if pos[0].size > 100:
                y1, y2 = pos[0].min(), pos[0].max()
                x1, x2 = pos[1].min(), pos[1].max()

                h, w = img_rgb.shape[:2]
                pad_y = max(1, int((y2 - y1 + 1) * 0.05))
                pad_x = max(1, int((x2 - x1 + 1) * 0.05))
                y1 = max(0, y1 - pad_y)
                y2 = min(h - 1, y2 + pad_y)
                x1 = max(0, x1 - pad_x)
                x2 = min(w - 1, x2 + pad_x)

                crop = img_rgb[y1:y2+1, x1:x2+1]

            del masks, scores

        if crop.size == 0:
            crop = img_rgb
        if crop is img_rgb:
            h, w = img_rgb.shape[:2]
            margin_h, margin_w = max(1, int(h * 0.2)), max(1, int(w * 0.2))
            crop = img_rgb[margin_h:h-margin_h, margin_w:w-margin_w]
            if crop.size == 0:
                crop = img_rgb

        crop_resized = cv2.resize(crop, (224, 224))
        imgs.append(crop_resized)

        del results, img_bgr, img_rgb, crop
        torch.cuda.empty_cache()

    if len(imgs) == 0:
        raise ValueError("Không tạo được crop nào từ ảnh tham chiếu.")

    embs = batch_embed(imgs)
    template = embs.mean(dim=0)
    result = torch.nn.functional.normalize(template, dim=0)

    del embs, template, imgs
    torch.cuda.empty_cache()

    return result

# def get_template(ref_paths):
#     imgs = [cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB) for p in ref_paths]
#     embs = batch_embed(imgs)
#     template = embs.mean(dim=0)
#     return torch.nn.functional.normalize(template, dim=0)

# ========================== HELPER FUNCTIONS ==========================
def bbox_center(bbox):
    return ((bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2)

def bbox_distance(bbox1, bbox2):
    c1 = bbox_center(bbox1)
    c2 = bbox_center(bbox2)
    return np.sqrt((c1[0] - c2[0])**2 + (c1[1] - c2[1])**2)

def bbox_iou(bbox1, bbox2):
    x1 = max(bbox1[0], bbox2[0])
    y1 = max(bbox1[1], bbox2[1])
    x2 = min(bbox1[2], bbox2[2])
    y2 = min(bbox1[3], bbox2[3])

    if x2 <= x1 or y2 <= y1:
        return 0.0

    inter = (x2 - x1) * (y2 - y1)
    area1 = (bbox1[2] - bbox1[0]) * (bbox1[3] - bbox1[1])
    area2 = (bbox2[2] - bbox2[0]) * (bbox2[3] - bbox2[1])
    union = area1 + area2 - inter

    return inter / union if union > 0 else 0.0

# ========================== MAIN PROCESS ==========================
@torch.inference_mode()
def process_video(folder: Path):
    video_path = folder / "drone_video_test_1.mp4"
    refs = sorted((folder / "object_images").glob("*.jpg"))
    video_id = folder.name

    template = get_template(refs, sam_template)

    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    detections = []
    skip_counter = 0

    last_bbox = None
    frames_since_last_det = 0
    max_frames_lost = 30

    pbar = tqdm(total=total_frames, desc=video_id, leave=False)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx = int(pbar.n)
        img_h, img_w = frame.shape[:2]

        should_detect = (skip_counter % 2 == 0)

        if should_detect:
            torch.cuda.empty_cache()

            results = sam(frame)[0]

            if results.masks is None or len(results.masks) == 0:
                frames_since_last_det += 1
                if frames_since_last_det > max_frames_lost:
                    last_bbox = None

                del results
                torch.cuda.empty_cache()

                pbar.update(1)
                skip_counter += 1
                continue

            masks = results.masks.data.cpu().numpy()

            del results
            torch.cuda.empty_cache()

            # Lọc masks theo pixel
            min_pixels = 70
            max_pixels = 4000
            valid_masks = []
            valid_areas = []

            for mask in masks:
                pos = np.where(mask)
                num_pixels = len(pos[0])
                if min_pixels <= num_pixels <= max_pixels:
                    valid_masks.append(mask)
                    valid_areas.append(num_pixels)

            if len(valid_masks) > 24:
                topk_idx = np.argsort(valid_areas)[-24:]
                masks = [valid_masks[i] for i in topk_idx]
            else:
                masks = valid_masks

            del valid_masks, valid_areas

            # Lọc bbox quá lớn
            max_bbox_width = img_w * 0.2
            max_bbox_height = img_h * 0.2
            max_bbox_area = (img_w * img_h) * 0.05

            size_filtered_masks = []
            for mask in masks:
                pos = np.where(mask)
                if pos[0].size == 0:
                    continue
                y1, y2 = pos[0].min(), pos[0].max()
                x1, x2 = pos[1].min(), pos[1].max()

                bbox_w = x2 - x1
                bbox_h = y2 - y1
                bbox_area = bbox_w * bbox_h

                if bbox_w > max_bbox_width or bbox_h > max_bbox_height or bbox_area > max_bbox_area:
                    continue

                size_filtered_masks.append(mask)

            masks = size_filtered_masks
            del size_filtered_masks

            # Crop và lấy bbox - THÊM KIỂM TRA
            crops_rgb = []
            bboxes = []
            for mask in masks:
                pos = np.where(mask)
                if pos[0].size == 0:
                    continue
                y1, y2 = pos[0].min(), pos[0].max()
                x1, x2 = pos[1].min(), pos[1].max()

                # KIỂM TRA tọa độ hợp lệ
                if y1 >= y2 or x1 >= x2:
                    continue

                # KIỂM TRA trong bounds
                if y1 < 0 or x1 < 0 or y2 >= img_h or x2 >= img_w:
                    continue

                crop = frame[y1:y2+1, x1:x2+1]

                # KIỂM TRA crop không rỗng
                if crop.size == 0 or crop.shape[0] == 0 or crop.shape[1] == 0:
                    continue

                try:
                    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                    crop_resized = cv2.resize(crop_rgb, (224, 224))
                    crops_rgb.append(crop_resized)
                    bboxes.append([x1, y1, x2, y2])
                except Exception as e:
                    # Bỏ qua nếu có lỗi
                    continue

            del masks

            if crops_rgb:
                embs = batch_embed(crops_rgb)
                sims = torch.nn.functional.cosine_similarity(embs, template, dim=1)

                sims_cpu = sims.cpu().numpy()
                del embs, sims
                torch.cuda.empty_cache()

                sim_threshold = 0.78
                candidates = []
                for idx in range(len(sims_cpu)):
                    if sims_cpu[idx] > sim_threshold:
                        candidates.append({
                            'idx': idx,
                            'sim': float(sims_cpu[idx]),
                            'bbox': bboxes[idx]
                        })

                del sims_cpu, crops_rgb, bboxes

                if candidates:
                    best_candidate = None

                    if last_bbox is None:
                        best_candidate = max(candidates, key=lambda x: x['sim'])
                    else:
                        max_velocity = 50 * (frames_since_last_det + 1)
                        max_distance = min(max_velocity, 200)

                        nearby_candidates = []
                        for cand in candidates:
                            dist = bbox_distance(last_bbox, cand['bbox'])
                            if dist <= max_distance:
                                proximity_score = 1 - (dist / max_distance)
                                combined_score = 0.6 * cand['sim'] + 0.4 * proximity_score
                                cand['combined_score'] = combined_score
                                cand['distance'] = dist
                                nearby_candidates.append(cand)

                        if nearby_candidates:
                            best_candidate = max(nearby_candidates, key=lambda x: x['combined_score'])
                        else:
                            high_sim_candidates = [c for c in candidates if c['sim'] > 0.85]
                            if high_sim_candidates:
                                best_candidate = max(high_sim_candidates, key=lambda x: x['sim'])
                                last_bbox = None

                    if best_candidate:
                        x1, y1, x2, y2 = best_candidate['bbox']
                        detections.append({
                            "frame": frame_idx,
                            "x1": int(x1),
                            "y1": int(y1),
                            "x2": int(x2),
                            "y2": int(y2)
                        })
                        last_bbox = best_candidate['bbox']
                        frames_since_last_det = 0
                    else:
                        frames_since_last_det += 1
                else:
                    frames_since_last_det += 1
            else:
                frames_since_last_det += 1

            if frames_since_last_det > max_frames_lost:
                last_bbox = None

        skip_counter += 1
        pbar.update(1)

        if frame_idx % 30 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    cap.release()
    pbar.close()

    gc.collect()
    torch.cuda.empty_cache()

    # INTERPOLATION
    if len(detections) > 1:
        full_dets = []
        max_gap = 5
        max_interp_distance = 100

        sorted_dets = sorted(detections, key=lambda x: x["frame"])

        for i in range(len(sorted_dets)):
            det = sorted_dets[i]
            full_dets.append(det)

            if i < len(sorted_dets) - 1:
                next_det = sorted_dets[i + 1]
                gap = next_det["frame"] - det["frame"]

                bbox1 = [det["x1"], det["y1"], det["x2"], det["y2"]]
                bbox2 = [next_det["x1"], next_det["y1"], next_det["x2"], next_det["y2"]]
                distance = bbox_distance(bbox1, bbox2)

                if 1 < gap <= max_gap and distance <= max_interp_distance:
                    for f in range(det["frame"] + 1, next_det["frame"]):
                        alpha = (f - det["frame"]) / gap
                        bbox = {
                            "frame": f,
                            "x1": int(det["x1"] * (1-alpha) + next_det["x1"] * alpha),
                            "y1": int(det["y1"] * (1-alpha) + next_det["y1"] * alpha),
                            "x2": int(det["x2"] * (1-alpha) + next_det["x2"] * alpha),
                            "y2": int(det["y2"] * (1-alpha) + next_det["y2"] * alpha),
                        }
                        full_dets.append(bbox)

        detections = sorted(full_dets, key=lambda x: x["frame"])

    if len(detections) > 0:
        final_output_detections = [{"bboxes": detections}]
    else:
        final_output_detections = []

    return {"video_id": video_id, "detections": final_output_detections}

# ========================== MAIN ==========================
if __name__ == "__main__":
    DATA_DIR = "/content/drive/MyDrive/ZaloAI/public_test/samples"
    OUTPUT_FILE = "/content/submission.json"

    results = []
    data_path = Path(DATA_DIR)

    if not data_path.exists():
        print(f"Lỗi: Không tìm thấy thư mục {DATA_DIR}")
    else:
        folders = sorted([f for f in data_path.iterdir() if (f / "drone_video_test_1.mp4").exists()])
        print(f"Tìm thấy {len(folders)} video để xử lý")

        for folder in folders:
            print(f"Processing {folder.name} ...")
            try:
                res = process_video(folder)
                results.append(res)
            except Exception as e:
                print(f"Error processing {folder.name}: {e}")
                results.append({"video_id": folder.name, "detections": []})

            gc.collect()
            torch.cuda.empty_cache()

        json.dump(results, open(OUTPUT_FILE, "w"), indent=2)
        print(f"\nHoàn tất! Output: {OUTPUT_FILE}")

        import shutil
        drive_output = f"{DATA_DIR}/submission.json"
        shutil.copy(OUTPUT_FILE, drive_output)
        print(f"Đã copy kết quả vào: {drive_output}")

Đang load FastSAM + DINOv2-small + float16...
Tìm thấy 1 video để xử lý
Processing BlackBox_0 ...


BlackBox_0:   0%|          | 0/440 [00:00<?, ?it/s]


Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8s-seg summary (fused): 85 layers, 11,779,987 parameters, 0 gradients, 39.9 GFLOPs
0: 640x640 1 object, 9.6ms
Speed: 2.3ms preprocess, 9.6ms inference, 2.8ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/segment/predict2


BlackBox_0:   0%|          | 2/440 [00:00<02:30,  2.91it/s]


0: 640x640 1 object, 9.9ms
Speed: 3.8ms preprocess, 9.9ms inference, 2.3ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/segment/predict2

0: 640x640 1 object, 9.6ms
Speed: 3.0ms preprocess, 9.6ms inference, 2.2ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/segment/predict2


...

BlackBox_0:  98%|█████████▊| 433/440 [00:43<00:00, 14.02it/s]


0: 640x640 1 object, 20.9ms
Speed: 3.5ms preprocess, 20.9ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/segment/predict2

0: 640x640 1 object, 21.2ms
Speed: 3.8ms preprocess, 21.2ms inference, 3.1ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/segment/predict2


BlackBox_0:  99%|█████████▉| 437/440 [00:44<00:00, 16.69it/s]


0: 640x640 1 object, 21.0ms
Speed: 4.2ms preprocess, 21.0ms inference, 3.3ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/segment/predict2



Hoàn tất! Output: /content/submission.json
Đã copy kết quả vào: /content/drive/MyDrive/ZaloAI/public_test/samples/submission.json


In [ ]:
# ========================== IMPORTS ==========================
import os
import gc
import cv2
import torch
import numpy as np
from pathlib import Path
import json
from tqdm import tqdm
from ultralytics.models.fastsam import FastSAMPredictor
from transformers import AutoImageProcessor, AutoModel

# ========================== LOAD MODELS ==========================
print("Đang load FastSAM + DINOv2-small + float16...")

overrides = dict(
    conf=0.25,
    task="segment",
    mode="predict",
    model="FastSAM-s.pt",
    imgsz=1024,
    retina_masks=False,
)
sam = FastSAMPredictor(overrides=overrides)

processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
dinov2 = AutoModel.from_pretrained("facebook/dinov2-small").cuda().half().eval()

# ========================== EMBEDDING BATCH ==========================
@torch.inference_mode()
def batch_embed(crops, batch_size=8):
    if len(crops) == 0:
        return torch.empty((0, 384), device="cuda")

    all_embs = []
    for i in range(0, len(crops), batch_size):
        batch_crops = crops[i:i+batch_size]
        inputs = processor(images=batch_crops, return_tensors="pt").to("cuda")

        with torch.autocast("cuda", dtype=torch.float16):
            out = dinov2(**inputs)

        patches = out.last_hidden_state[:, 1:]
        topk = patches.topk(16, dim=1).values.mean(dim=1)
        embs = torch.nn.functional.normalize(topk, dim=1)

        # Chuyển về CPU ngay
        all_embs.append(embs.cpu())

        # Xóa từng biến
        del inputs, out, patches, topk, embs

    result = torch.cat(all_embs, dim=0).cuda()
    del all_embs
    torch.cuda.empty_cache()

    return result

# ========================== REFERENCE TEMPLATE ==========================
def get_template(ref_paths, sam_predictor):
    imgs = []
    for p in ref_paths:
        img_bgr = cv2.imread(str(p))
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # Resize ảnh tham chiếu xuống để tiết kiệm RAM
        h, w = img_rgb.shape[:2]
        if max(h, w) > 1024:
            scale = 1024 / max(h, w)
            new_h, new_w = int(h * scale), int(w * scale)
            img_rgb = cv2.resize(img_rgb, (new_w, new_h))

        crop = img_rgb

        results = sam_predictor(img_rgb)[0]

        if results.masks is not None and len(results.masks) > 0:
            masks = results.masks.data.cpu().numpy()

            scores = None
            if results.boxes is not None and results.boxes.conf is not None:
                scores = results.boxes.conf.detach().cpu().numpy()
            elif hasattr(results.masks, "scores") and results.masks.scores is not None:
                scores = results.masks.scores.detach().cpu().numpy()

            if scores is not None and len(scores) == len(masks):
                best_idx = int(np.argmax(scores))
            else:
                areas = masks.sum(axis=(1, 2))
                best_idx = int(np.argmax(areas))

            best_mask = masks[best_idx]
            pos = np.where(best_mask)

            if pos[0].size > 100:
                y1, y2 = pos[0].min(), pos[0].max()
                x1, x2 = pos[1].min(), pos[1].max()

                h, w = img_rgb.shape[:2]
                pad_y = max(1, int((y2 - y1 + 1) * 0.05))
                pad_x = max(1, int((x2 - x1 + 1) * 0.05))
                y1 = max(0, y1 - pad_y)
                y2 = min(h - 1, y2 + pad_y)
                x1 = max(0, x1 - pad_x)
                x2 = min(w - 1, x2 + pad_x)

                crop = img_rgb[y1:y2+1, x1:x2+1]

            del masks, scores

        if crop.size == 0:
            crop = img_rgb
        if crop is img_rgb:
            h, w = img_rgb.shape[:2]
            margin_h, margin_w = max(1, int(h * 0.2)), max(1, int(w * 0.2))
            crop = img_rgb[margin_h:h-margin_h, margin_w:w-margin_w]
            if crop.size == 0:
                crop = img_rgb

        crop_resized = cv2.resize(crop, (224, 224))
        imgs.append(crop_resized)

        del results, img_bgr, img_rgb, crop
        torch.cuda.empty_cache()

    if len(imgs) == 0:
        raise ValueError("Không tạo được crop nào từ ảnh tham chiếu.")

    embs = batch_embed(imgs)
    template = embs.mean(dim=0)
    result = torch.nn.functional.normalize(template, dim=0)

    del embs, template, imgs
    torch.cuda.empty_cache()

    return result

# ========================== HELPER FUNCTIONS ==========================
def bbox_center(bbox):
    return ((bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2)

def bbox_distance(bbox1, bbox2):
    c1 = bbox_center(bbox1)
    c2 = bbox_center(bbox2)
    return np.sqrt((c1[0] - c2[0])**2 + (c1[1] - c2[1])**2)

def bbox_iou(bbox1, bbox2):
    x1 = max(bbox1[0], bbox2[0])
    y1 = max(bbox1[1], bbox2[1])
    x2 = min(bbox1[2], bbox2[2])
    y2 = min(bbox1[3], bbox2[3])

    if x2 <= x1 or y2 <= y1:
        return 0.0

    inter = (x2 - x1) * (y2 - y1)
    area1 = (bbox1[2] - bbox1[0]) * (bbox1[3] - bbox1[1])
    area2 = (bbox2[2] - bbox2[0]) * (bbox2[3] - bbox2[1])
    union = area1 + area2 - inter

    return inter / union if union > 0 else 0.0

# ========================== PROCESS SINGLE FRAME ==========================
def process_frame(frame, template, sam_predictor, img_h, img_w):
    """Xử lý 1 frame và trả về candidates"""

    # Clear cache trước khi bắt đầu
    torch.cuda.empty_cache()

    results = sam_predictor(frame)[0]

    if results.masks is None or len(results.masks) == 0:
        del results
        torch.cuda.empty_cache()
        return []

    # Chuyển về numpy và xóa tensor ngay
    masks = results.masks.data.cpu().numpy()
    del results
    torch.cuda.empty_cache()

    # Lọc masks theo pixel
    min_pixels = 70
    max_pixels = 4000
    valid_masks = []
    valid_areas = []

    for mask in masks:
        pos = np.where(mask)
        num_pixels = len(pos[0])
        if min_pixels <= num_pixels <= max_pixels:
            valid_masks.append(mask)
            valid_areas.append(num_pixels)

    # Top 12 thay vì 16 để tiết kiệm hơn
    if len(valid_masks) > 24:
        topk_idx = np.argsort(valid_areas)[-24:]
        masks = [valid_masks[i] for i in topk_idx]
    else:
        masks = valid_masks

    del valid_masks, valid_areas

    # Lọc bbox quá lớn
    max_bbox_width = img_w * 0.2
    max_bbox_height = img_h * 0.2
    max_bbox_area = (img_w * img_h) * 0.05

    size_filtered_masks = []
    for mask in masks:
        pos = np.where(mask)
        if pos[0].size == 0:
            continue
        y1, y2 = pos[0].min(), pos[0].max()
        x1, x2 = pos[1].min(), pos[1].max()

        bbox_w = x2 - x1
        bbox_h = y2 - y1
        bbox_area = bbox_w * bbox_h

        if bbox_w > max_bbox_width or bbox_h > max_bbox_height or bbox_area > max_bbox_area:
            continue

        size_filtered_masks.append(mask)

    masks = size_filtered_masks
    del size_filtered_masks

    # Crop và lấy bbox
    crops_rgb = []
    bboxes = []
    for mask in masks:
        pos = np.where(mask)
        if pos[0].size == 0:
            continue
        y1, y2 = pos[0].min(), pos[0].max()
        x1, x2 = pos[1].min(), pos[1].max()

        crop = frame[y1:y2+1, x1:x2+1]
        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        crops_rgb.append(cv2.resize(crop_rgb, (224, 224)))
        bboxes.append([x1, y1, x2, y2])

    del masks

    if not crops_rgb:
        return []

    # Embed và tính similarity
    embs = batch_embed(crops_rgb)
    sims = torch.nn.functional.cosine_similarity(embs, template, dim=1)

    # Chuyển về numpy
    sims_cpu = sims.cpu().numpy()
    del embs, sims
    torch.cuda.empty_cache()

    # Tạo candidates
    sim_threshold = 0.82
    candidates = []
    for idx in range(len(sims_cpu)):
        if sims_cpu[idx] > sim_threshold:
            candidates.append({
                'idx': idx,
                'sim': float(sims_cpu[idx]),
                'bbox': bboxes[idx]
            })

    del sims_cpu, crops_rgb, bboxes

    return candidates

# ========================== MAIN PROCESS ==========================
@torch.inference_mode()
def process_video(folder: Path):
    video_path = folder / "drone_video_test_1.mp4"
    refs = sorted((folder / "object_images").glob("*.jpg"))
    video_id = folder.name

    template = get_template(refs, sam)

    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    detections = []
    skip_counter = 0

    last_bbox = None
    frames_since_last_det = 0
    max_frames_lost = 30

    pbar = tqdm(total=total_frames, desc=video_id, leave=False)

    # Adaptive skip based on video length
    skip_interval = 2

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx = int(pbar.n)
        img_h, img_w = frame.shape[:2]

        # Skip frames
        should_detect = (skip_counter % skip_interval == 0)

        if should_detect:
            # Resize frame nếu quá lớn
            process_frame_img = frame
            scale = 1.0
            if max(img_h, img_w) > 1280:
                scale = 1280 / max(img_h, img_w)
                new_h, new_w = int(img_h * scale), int(img_w * scale)
                process_frame_img = cv2.resize(frame, (new_w, new_h))

            # Process frame
            candidates = process_frame(process_frame_img, template, sam,
                                      int(img_h * scale), int(img_w * scale))

            # Scale bboxes back if resized
            if scale != 1.0:
                for cand in candidates:
                    bbox = cand['bbox']
                    cand['bbox'] = [int(x / scale) for x in bbox]

            if candidates:
                best_candidate = None

                if last_bbox is None:
                    best_candidate = max(candidates, key=lambda x: x['sim'])
                else:
                    max_velocity = 50 * (frames_since_last_det + 1)
                    max_distance = min(max_velocity, 200)

                    nearby_candidates = []
                    for cand in candidates:
                        dist = bbox_distance(last_bbox, cand['bbox'])
                        if dist <= max_distance:
                            proximity_score = 1 - (dist / max_distance)
                            combined_score = 0.6 * cand['sim'] + 0.4 * proximity_score
                            cand['combined_score'] = combined_score
                            cand['distance'] = dist
                            nearby_candidates.append(cand)

                    if nearby_candidates:
                        best_candidate = max(nearby_candidates, key=lambda x: x['combined_score'])
                    else:
                        high_sim_candidates = [c for c in candidates if c['sim'] > 0.85]
                        if high_sim_candidates:
                            best_candidate = max(high_sim_candidates, key=lambda x: x['sim'])
                            last_bbox = None

                if best_candidate:
                    x1, y1, x2, y2 = best_candidate['bbox']
                    detections.append({
                        "frame": frame_idx,
                        "x1": int(x1),
                        "y1": int(y1),
                        "x2": int(x2),
                        "y2": int(y2)
                    })
                    last_bbox = best_candidate['bbox']
                    frames_since_last_det = 0
                else:
                    frames_since_last_det += 1
            else:
                frames_since_last_det += 1

            if frames_since_last_det > max_frames_lost:
                last_bbox = None

            # Clear cache sau mỗi lần detect
            torch.cuda.empty_cache()

        else:
            frames_since_last_det += 1

        skip_counter += 1
        pbar.update(1)

        # Aggressive cleanup mỗi 20 frames
        if frame_idx % 20 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    cap.release()
    pbar.close()

    # Final cleanup
    del template
    gc.collect()
    torch.cuda.empty_cache()

    # INTERPOLATION
    if len(detections) > 1:
        full_dets = []
        max_gap = 5  # Tăng lên vì skip nhiều hơn
        max_interp_distance = 80

        sorted_dets = sorted(detections, key=lambda x: x["frame"])

        for i in range(len(sorted_dets)):
            det = sorted_dets[i]
            full_dets.append(det)

            if i < len(sorted_dets) - 1:
                next_det = sorted_dets[i + 1]
                gap = next_det["frame"] - det["frame"]

                bbox1 = [det["x1"], det["y1"], det["x2"], det["y2"]]
                bbox2 = [next_det["x1"], next_det["y1"], next_det["x2"], next_det["y2"]]
                distance = bbox_distance(bbox1, bbox2)

                if 1 < gap <= max_gap and distance <= max_interp_distance:
                    for f in range(det["frame"] + 1, next_det["frame"]):
                        alpha = (f - det["frame"]) / gap
                        bbox = {
                            "frame": f,
                            "x1": int(det["x1"] * (1-alpha) + next_det["x1"] * alpha),
                            "y1": int(det["y1"] * (1-alpha) + next_det["y1"] * alpha),
                            "x2": int(det["x2"] * (1-alpha) + next_det["x2"] * alpha),
                            "y2": int(det["y2"] * (1-alpha) + next_det["y2"] * alpha),
                        }
                        full_dets.append(bbox)

        detections = sorted(full_dets, key=lambda x: x["frame"])

    if len(detections) > 0:
        final_output_detections = [{"bboxes": detections}]
    else:
        final_output_detections = []

    return {"video_id": video_id, "detections": final_output_detections}

# ========================== MAIN ==========================
if __name__ == "__main__":
    DATA_DIR = "/content/drive/MyDrive/ZaloAI/public_test/samples"
    OUTPUT_FILE = "/content/submission.json"

    results = []
    data_path = Path(DATA_DIR)

    if not data_path.exists():
        print(f"Lỗi: Không tìm thấy thư mục {DATA_DIR}")
    else:
        folders = sorted([f for f in data_path.iterdir() if (f / "drone_video_test_1.mp4").exists()])
        print(f"Tìm thấy {len(folders)} video để xử lý")

        for idx, folder in enumerate(folders):
            print(f"Processing {folder.name} ({idx+1}/{len(folders)})...")

            try:
                res = process_video(folder)
                results.append(res)
            except Exception as e:
                print(f"Error processing {folder.name}: {e}")
                results.append({"video_id": folder.name, "detections": []})

            # Aggressive cleanup sau mỗi video
            gc.collect()
            torch.cuda.empty_cache()

            # Log memory usage
            if torch.cuda.is_available():
                mem_allocated = torch.cuda.memory_allocated() / 1024**3
                mem_reserved = torch.cuda.memory_reserved() / 1024**3
                print(f"GPU Memory: {mem_allocated:.2f}GB allocated, {mem_reserved:.2f}GB reserved")

        json.dump(results, open(OUTPUT_FILE, "w"), indent=2)
        print(f"\nHoàn tất! Output: {OUTPUT_FILE}")

        import shutil
        drive_output = f"{DATA_DIR}/submission.json"
        shutil.copy(OUTPUT_FILE, drive_output)
        print(f"Đã copy kết quả vào: {drive_output}")

Đang load FastSAM + DINOv2-small + float16...
Tìm thấy 1 video để xử lý
Processing BlackBox_0 (1/1)...

Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
YOLOv8s-seg summary (fused): 85 layers, 11,779,987 parameters, 0 gradients, 39.9 GFLOPs
0: 1024x1024 145 objects, 8.4ms
Speed: 6.7ms preprocess, 8.4ms inference, 7.5ms postprocess per image at shape (1, 3, 1024, 1024)
Results saved to /content/runs/segment/predict13

0: 1024x1024 226 objects, 10.3ms
Speed: 7.8ms preprocess, 10.3ms inference, 9.8ms postprocess per image at shape (1, 3, 1024, 1024)
Results saved to /content/runs/segment/predict13

0: 1024x1024 158 objects, 10.2ms
Speed: 7.9ms preprocess, 10.2ms inference, 7.5ms postprocess per image at shape (1, 3, 1024, 1024)
Results saved to /content/runs/segment/predict13



BlackBox_0:   0%|          | 0/440 [00:00<?, ?it/s]


0: 1024x1024 1 object, 8.8ms
Speed: 3.9ms preprocess, 8.8ms inference, 2.5ms postprocess per image at shape (1, 3, 1024, 1024)
Results saved to /content/runs/segment/predict13



BlackBox_0:   0%|          | 2/440 [00:00<01:29,  4.88it/s]


0: 1024x1024 1 object, 11.0ms
Speed: 3.9ms preprocess, 11.0ms inference, 2.8ms postprocess per image at shape (1, 3, 1024, 1024)
Results saved to /content/runs/segment/predict13

0: 1024x1024 1 object, 9.4ms
Speed: 3.6ms preprocess, 9.4ms inference, 2.5ms postprocess per image at shape (1, 3, 1024, 1024)
Results saved to /content/runs/segment/predict13


...


BlackBox_0:  16%|█▌        | 71/440 [00:04<00:24, 14.92it/s]


0: 1024x1024 11 objects, 10.0ms
Speed: 3.6ms preprocess, 10.0ms inference, 3.0ms postprocess per image at shape (1, 3, 1024, 1024)
Results saved to /content/runs/segment/predict13



BlackBox_0:  17%|█▋        | 73/440 [00:04<00:24, 14.96it/s]


0: 1024x1024 10 objects, 9.7ms
Speed: 5.0ms preprocess, 9.7ms inference, 2.7ms postprocess per image at shape (1, 3, 1024, 1024)
Results saved to /content/runs/segment/predict13


Error processing BlackBox_0: OpenCV(4.12.0) /io/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'

GPU Memory: 0.11GB allocated, 0.25GB reserved

Hoàn tất! Output: /content/submission.json
Đã copy kết quả vào: /content/drive/MyDrive/ZaloAI/public_test/samples/submission.json


In [ ]:
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ========================== IMPORTS ==========================
import os
import gc
import cv2
import torch
import numpy as np
from pathlib import Path
import json
from tqdm import tqdm
from ultralytics.models.fastsam import FastSAMPredictor
from ultralytics import SAM
from transformers import AutoImageProcessor, AutoModel


# ========================== LOAD MODELS ==========================
print("Đang load FastSAM + DINOv2-small + float16...")

# Giữ nguyên Predictor
overrides = dict(
    conf=0.25,
    task="segment",
    mode="predict",
    model="FastSAM-s.pt",
    imgsz=640,
    retina_masks=False,
    max_det=50
)

sam_template = SAM("mobile_sam.pt")
sam = FastSAMPredictor(overrides=overrides)

processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
dinov2 = AutoModel.from_pretrained("facebook/dinov2-small").cuda().half().eval()

print("✅ Models loaded successfully!")

# ========================== EMBEDDING BATCH ==========================
@torch.inference_mode()
def batch_embed(crops, batch_size=4):
    if len(crops) == 0:
        return torch.empty((0, 384), device="cuda")

    all_embs = []
    for i in range(0, len(crops), batch_size):
        batch_crops = crops[i:i+batch_size]
        inputs = processor(images=batch_crops, return_tensors="pt").to("cuda")

        with torch.autocast("cuda", dtype=torch.float16):
            out = dinov2(**inputs)

        patches = out.last_hidden_state[:, 1:]
        topk = patches.topk(16, dim=1).values.mean(dim=1)
        embs = torch.nn.functional.normalize(topk, dim=1)

        all_embs.append(embs.cpu())

        del inputs, out, patches, topk, embs

    result = torch.cat(all_embs, dim=0).cuda()
    del all_embs
    torch.cuda.empty_cache()

    return result

# ========================== REFERENCE TEMPLATE ==========================
def get_template(ref_paths, sam_model):
    print(f"\n🔍 Processing {len(ref_paths)} reference images...")
    imgs = []

    for idx, p in enumerate(ref_paths):
        print(f"  📸 Image {idx+1}/{len(ref_paths)}: {p.name}")

        img_bgr = cv2.imread(str(p))
        if img_bgr is None:
            print(f"    ⚠️  Failed to read image")
            continue

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        print(f"    📐 Original size: {img_rgb.shape[1]}x{img_rgb.shape[0]}")

        crop = img_rgb

        results = sam_model(img_rgb)
        result = results[0] if len(results) > 0 else None

        if result is not None and result.masks is not None and len(result.masks) > 0:
            masks = result.masks.data.cpu().numpy()
            print(f"    🎭 Found {len(masks)} masks")

            scores = None
            if result.boxes is not None and result.boxes.conf is not None:
                scores = result.boxes.conf.detach().cpu().numpy()
            elif hasattr(result.masks, "scores") and result.masks.scores is not None:
                scores = result.masks.scores.detach().cpu().numpy()

            if scores is not None and len(scores) == len(masks):
                best_idx = int(np.argmax(scores))
                print(f"    ⭐ Best mask: #{best_idx} (score: {scores[best_idx]:.3f})")
            else:
                areas = masks.sum(axis=(1, 2))
                best_idx = int(np.argmax(areas))
                print(f"    ⭐ Best mask: #{best_idx} (largest area: {int(areas[best_idx])} pixels)")

            best_mask = masks[best_idx]
            pos = np.where(best_mask)

            if pos[0].size > 100:
                y1, y2 = pos[0].min(), pos[0].max()
                x1, x2 = pos[1].min(), pos[1].max()

                h, w = img_rgb.shape[:2]
                pad_y = max(1, int((y2 - y1 + 1) * 0.05))
                pad_x = max(1, int((x2 - x1 + 1) * 0.05))
                y1 = max(0, y1 - pad_y)
                y2 = min(h - 1, y2 + pad_y)
                x1 = max(0, x1 - pad_x)
                x2 = min(w - 1, x2 + pad_x)

                crop = img_rgb[y1:y2+1, x1:x2+1]
                print(f"    ✂️  Cropped to: {crop.shape[1]}x{crop.shape[0]} (bbox: [{x1},{y1},{x2},{y2}])")

            del masks, scores
        else:
            print(f"    ⚠️  No masks found, using fallback crop")

        if crop.size == 0:
            crop = img_rgb
        if crop is img_rgb:
            h, w = img_rgb.shape[:2]
            margin_h, margin_w = max(1, int(h * 0.2)), max(1, int(w * 0.2))
            crop = img_rgb[margin_h:h-margin_h, margin_w:w-margin_w]
            if crop.size == 0:
                crop = img_rgb
            print(f"    ✂️  Center crop: {crop.shape[1]}x{crop.shape[0]}")

        crop_resized = cv2.resize(crop, (224, 224))
        imgs.append(crop_resized)
        print(f"    ✅ Resized to: 224x224")

        del results, result, img_bgr, img_rgb, crop
        torch.cuda.empty_cache()

    if len(imgs) == 0:
        raise ValueError("❌ Không tạo được crop nào từ ảnh tham chiếu.")

    print(f"\n🧠 Embedding {len(imgs)} crops...")
    embs = batch_embed(imgs)
    print(f"✅ Embeddings shape: {embs.shape}")

    template = embs.mean(dim=0)
    result = torch.nn.functional.normalize(template, dim=0)

    print(f"✅ Template created! Shape: {result.shape}, Norm: {torch.norm(result).item():.4f}")

    del embs, template, imgs
    torch.cuda.empty_cache()

    return result

# ========================== HELPER FUNCTIONS ==========================
def bbox_center(bbox):
    return ((bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2)

def bbox_distance(bbox1, bbox2):
    c1 = bbox_center(bbox1)
    c2 = bbox_center(bbox2)
    return np.sqrt((c1[0] - c2[0])**2 + (c1[1] - c2[1])**2)

def bbox_iou(bbox1, bbox2):
    x1 = max(bbox1[0], bbox2[0])
    y1 = max(bbox1[1], bbox2[1])
    x2 = min(bbox1[2], bbox2[2])
    y2 = min(bbox1[3], bbox2[3])

    if x2 <= x1 or y2 <= y1:
        return 0.0

    inter = (x2 - x1) * (y2 - y1)
    area1 = (bbox1[2] - bbox1[0]) * (bbox1[3] - bbox1[1])
    area2 = (bbox2[2] - bbox2[0]) * (bbox2[3] - bbox2[1])
    union = area1 + area2 - inter

    return inter / union if union > 0 else 0.0

# ========================== MAIN PROCESS ==========================
@torch.inference_mode()
def process_video(folder: Path):
    video_path = folder / "drone_video_test_1.mp4"
    refs = sorted((folder / "object_images").glob("*.jpg"))
    video_id = folder.name

    print(f"\n{'='*60}")
    print(f"📹 Video: {video_id}")
    print(f"{'='*60}")

    template = get_template(refs, sam_template)

    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"\n🎬 Processing video: {total_frames} frames")

    detections = []
    skip_counter = 0

    last_bbox = None
    frames_since_last_det = 0
    max_frames_lost = 30

    pbar = tqdm(total=total_frames, desc=video_id, leave=False)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx = int(pbar.n)
        img_h, img_w = frame.shape[:2]

        should_detect = (skip_counter % 2 == 0)

        if should_detect:
            torch.cuda.empty_cache()

            results = sam(frame)[0]

            # ✅ KIỂM TRA CÓ BOXES VÀ MASKS
            if results.masks is None or len(results.masks) == 0:
                frames_since_last_det += 1
                if frames_since_last_det > max_frames_lost:
                    last_bbox = None

                del results
                torch.cuda.empty_cache()

                pbar.update(1)
                skip_counter += 1
                continue

            # ✅ LẤY BOXES.XYXY (TỌA ĐỘ FRAME GỐC) VÀ MASKS
            if results.boxes is not None and results.boxes.xyxy is not None:
                boxes_xyxy = results.boxes.xyxy.cpu().numpy()
                has_boxes = True
            else:
                boxes_xyxy = None
                has_boxes = False

            masks = results.masks.data.cpu().numpy()

            if results.boxes is not None and results.boxes.conf is not None:
                confidences = results.boxes.conf.cpu().numpy()
            else:
                confidences = None

            del results
            torch.cuda.empty_cache()

            # ✅ Lọc masks theo pixel (VẪN DÙNG MASKS)
            min_pixels = 70
            max_pixels = 4000
            valid_mask_indices = []
            valid_areas = []
            valid_confidences = []

            for idx, mask in enumerate(masks):
                pos = np.where(mask)
                num_pixels = len(pos[0])
                if min_pixels <= num_pixels <= max_pixels:
                    valid_mask_indices.append(idx)
                    valid_areas.append(num_pixels)
                    if confidences is not None and idx < len(confidences):
                        valid_confidences.append(confidences[idx])

            # ✅ Lấy top 24 theo confidence hoặc area
            if len(valid_mask_indices) > 24:
                if valid_confidences and len(valid_confidences) == len(valid_mask_indices):
                    topk_idx = np.argsort(valid_confidences)[::-1][:24]
                    valid_mask_indices = [valid_mask_indices[i] for i in topk_idx]
                    valid_areas = [valid_areas[i] for i in topk_idx]
                    valid_confidences = [valid_confidences[i] for i in topk_idx]
                else:
                    topk_idx = np.argsort(valid_areas)[::-1][:24]
                    valid_mask_indices = [valid_mask_indices[i] for i in topk_idx]
                    valid_areas = [valid_areas[i] for i in topk_idx]

            del masks, confidences

            # ✅ Lọc bbox quá lớn
            max_bbox_width = img_w * 0.2
            max_bbox_height = img_h * 0.2
            max_bbox_area = (img_w * img_h) * 0.05

            size_filtered_indices = []
            size_filtered_areas = []
            size_filtered_confidences = []

            for i, idx in enumerate(valid_mask_indices):
                if has_boxes and idx < len(boxes_xyxy):
                    # ✅ DÙNG BOXES.XYXY (ĐÃ Ở TỌA ĐỘ FRAME GỐC)
                    bbox = boxes_xyxy[idx]
                    x1, y1, x2, y2 = bbox
                else:
                    # Fallback: tính từ mask (cần scale)
                    continue  # Bỏ qua nếu không có boxes

                bbox_w = x2 - x1
                bbox_h = y2 - y1
                bbox_area = bbox_w * bbox_h

                if bbox_w > max_bbox_width or bbox_h > max_bbox_height or bbox_area > max_bbox_area:
                    continue

                size_filtered_indices.append(idx)
                size_filtered_areas.append(valid_areas[i])
                if valid_confidences and i < len(valid_confidences):
                    size_filtered_confidences.append(valid_confidences[i])

            del valid_mask_indices, valid_areas, valid_confidences

            # ✅ Crop và embed - DÙNG BOXES.XYXY
            crops_rgb = []
            bboxes = []

            for idx in size_filtered_indices:
                if has_boxes and idx < len(boxes_xyxy):
                    bbox = boxes_xyxy[idx]
                    x1, y1, x2, y2 = bbox

                    # Chuyển sang int và clamp
                    x1 = max(0, int(x1))
                    y1 = max(0, int(y1))
                    x2 = min(img_w - 1, int(x2))
                    y2 = min(img_h - 1, int(y2))

                    # KIỂM TRA tọa độ hợp lệ
                    if x1 >= x2 or y1 >= y2:
                        continue

                    # ✅ CROP TRỰC TIẾP TỪ FRAME (bbox đã ở tọa độ frame gốc)
                    crop = frame[y1:y2+1, x1:x2+1]

                    if crop.size == 0 or crop.shape[0] == 0 or crop.shape[1] == 0:
                        continue

                    try:
                        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                        crop_resized = cv2.resize(crop_rgb, (224, 224))
                        crops_rgb.append(crop_resized)
                        bboxes.append([x1, y1, x2, y2])
                    except Exception as e:
                        continue

            del size_filtered_indices, size_filtered_areas, size_filtered_confidences

            if crops_rgb:
                embs = batch_embed(crops_rgb)
                sims = torch.nn.functional.cosine_similarity(embs, template, dim=1)

                sims_cpu = sims.cpu().numpy()
                del embs, sims
                torch.cuda.empty_cache()

                sim_threshold = 0.82
                candidates = []
                for idx in range(len(sims_cpu)):
                    if sims_cpu[idx] > sim_threshold:
                        candidates.append({
                            'idx': idx,
                            'sim': float(sims_cpu[idx]),
                            'bbox': bboxes[idx]
                        })

                del sims_cpu, crops_rgb, bboxes

                if candidates:
                    best_candidate = None

                    if last_bbox is None:
                        best_candidate = max(candidates, key=lambda x: x['sim'])
                    else:
                        max_velocity = 50 * (frames_since_last_det + 1)
                        max_distance = min(max_velocity, 200)

                        nearby_candidates = []
                        for cand in candidates:
                            dist = bbox_distance(last_bbox, cand['bbox'])
                            if dist <= max_distance:
                                proximity_score = 1 - (dist / max_distance)
                                combined_score = 0.6 * cand['sim'] + 0.4 * proximity_score
                                cand['combined_score'] = combined_score
                                cand['distance'] = dist
                                nearby_candidates.append(cand)

                        if nearby_candidates:
                            best_candidate = max(nearby_candidates, key=lambda x: x['combined_score'])
                        else:
                            high_sim_candidates = [c for c in candidates if c['sim'] > 0.85]
                            if high_sim_candidates:
                                best_candidate = max(high_sim_candidates, key=lambda x: x['sim'])
                                last_bbox = None

                    if best_candidate:
                        x1, y1, x2, y2 = best_candidate['bbox']
                        detections.append({
                            "frame": frame_idx,
                            "x1": int(x1),
                            "y1": int(y1),
                            "x2": int(x2),
                            "y2": int(y2)
                        })
                        last_bbox = best_candidate['bbox']
                        frames_since_last_det = 0
                    else:
                        frames_since_last_det += 1
                else:
                    frames_since_last_det += 1
            else:
                frames_since_last_det += 1

            if frames_since_last_det > max_frames_lost:
                last_bbox = None

        skip_counter += 1
        pbar.update(1)

        if frame_idx % 30 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    cap.release()
    pbar.close()

    print(f"✅ Detected {len(detections)} bboxes before interpolation")

    gc.collect()
    torch.cuda.empty_cache()

    # INTERPOLATION
    if len(detections) > 1:
        full_dets = []
        max_gap = 5
        max_interp_distance = 100

        sorted_dets = sorted(detections, key=lambda x: x["frame"])

        for i in range(len(sorted_dets)):
            det = sorted_dets[i]
            full_dets.append(det)

            if i < len(sorted_dets) - 1:
                next_det = sorted_dets[i + 1]
                gap = next_det["frame"] - det["frame"]

                bbox1 = [det["x1"], det["y1"], det["x2"], det["y2"]]
                bbox2 = [next_det["x1"], next_det["y1"], next_det["x2"], next_det["y2"]]
                distance = bbox_distance(bbox1, bbox2)

                if 1 < gap <= max_gap and distance <= max_interp_distance:
                    for f in range(det["frame"] + 1, next_det["frame"]):
                        alpha = (f - det["frame"]) / gap
                        bbox = {
                            "frame": f,
                            "x1": int(det["x1"] * (1-alpha) + next_det["x1"] * alpha),
                            "y1": int(det["y1"] * (1-alpha) + next_det["y1"] * alpha),
                            "x2": int(det["x2"] * (1-alpha) + next_det["x2"] * alpha),
                            "y2": int(det["y2"] * (1-alpha) + next_det["y2"] * alpha),
                        }
                        full_dets.append(bbox)

        detections = sorted(full_dets, key=lambda x: x["frame"])
        print(f"✅ After interpolation: {len(detections)} bboxes")

    if len(detections) > 0:
        final_output_detections = [{"bboxes": detections}]
    else:
        final_output_detections = []

    return {"video_id": video_id, "detections": final_output_detections}

# ========================== MAIN ==========================
if __name__ == "__main__":
    DATA_DIR = "/content/drive/MyDrive/ZaloAI/public_test/samples"
    OUTPUT_FILE = "/content/submission.json"

    results = []
    data_path = Path(DATA_DIR)

    if not data_path.exists():
        print(f"❌ Lỗi: Không tìm thấy thư mục {DATA_DIR}")
    else:
        folders = sorted([f for f in data_path.iterdir() if (f / "drone_video_test_1.mp4").exists()])
        print(f"\n🎯 Tìm thấy {len(folders)} video để xử lý\n")

        for folder in folders:
            try:
                res = process_video(folder)
                results.append(res)
            except Exception as e:
                print(f"❌ Error processing {folder.name}: {e}")
                import traceback
                traceback.print_exc()
                results.append({"video_id": folder.name, "detections": []})

            gc.collect()
            torch.cuda.empty_cache()

        json.dump(results, open(OUTPUT_FILE, "w"), indent=2)
        print(f"\n✅ Hoàn tất! Output: {OUTPUT_FILE}")

        import shutil
        drive_output = f"{DATA_DIR}/submission.json"
        shutil.copy(OUTPUT_FILE, drive_output)
        print(f"✅ Đã copy kết quả vào: {drive_output}")

ModuleNotFoundError: No module named 'ultralytics'

In [ ]:
from ultralytics.models.fastsam import FastSAMPredictor

# Create FastSAMPredictor
overrides = dict(conf=0.25, task="segment", retina_masks=False, mode="predict", model="FastSAM-s.pt", save=True, imgsz=640)
predictor = FastSAMPredictor(overrides=overrides)

# Segment everything
image_path = "drone-blackbox-13.png"
everything_results = predictor(image_path)


Ultralytics 8.3.230 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8s-seg summary (fused): 85 layers, 11,779,987 parameters, 0 gradients, 39.9 GFLOPs
image 1/1 /content/drone-blackbox-13.png: 640x640 8 objects, 20.8ms
Speed: 2.3ms preprocess, 20.8ms inference, 5.5ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/segment/predict2


In [ ]:
import cv2
import numpy as np
from transformers import AutoImageProcessor, ConvNextV2Model
import torch
from pathlib import Path
from ultralytics.models.fastsam import FastSAMPredictor

overrides = dict(
    conf=0.25,
    task="segment",
    mode="predict",
    model="FastSAM-s.pt",
    imgsz=640,
    retina_masks=False
)
sam = FastSAMPredictor(overrides=overrides)

processor = AutoImageProcessor.from_pretrained("facebook/convnextv2-nano-22k-224")
convnext = ConvNextV2Model.from_pretrained("facebook/convnextv2-nano-22k-224").cuda().eval()

print("✅ Loaded ConvNeXtV2-Nano: 15M params")

# ========================== DEBUG FIRST ==========================
@torch.inference_mode()
def debug_outputs():
    """Kiểm tra tất cả output có thể dùng"""
    test_img = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
    inputs = processor(images=[test_img], return_tensors="pt").to("cuda")

    outputs = convnext(**inputs, output_hidden_states=True)

    print("\n" + "="*60)
    print("ALL AVAILABLE OUTPUTS:")
    print("="*60)

    # 1. Pooler output
    if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
        pooler = outputs.pooler_output
        print(f"\n1. pooler_output:")
        print(f"   Shape: {pooler.shape}")
        print(f"   Mean: {pooler.mean().item():.4f}")
        print(f"   Std: {pooler.std().item():.4f}")
        print(f"   Range: [{pooler.min().item():.4f}, {pooler.max().item():.4f}]")

    # 2. Last hidden state
    if hasattr(outputs, 'last_hidden_state'):
        last_hidden = outputs.last_hidden_state
        print(f"\n2. last_hidden_state:")
        print(f"   Shape: {last_hidden.shape}")
        print(f"   Mean: {last_hidden.mean().item():.4f}")
        print(f"   Std: {last_hidden.std().item():.4f}")
        print(f"   Range: [{last_hidden.min().item():.4f}, {last_hidden.max().item():.4f}]")

        # Test manual pooling
        pooled = last_hidden.mean(dim=[2, 3])
        print(f"\n   Manual pooling (mean):")
        print(f"   Shape: {pooled.shape}")
        print(f"   Mean: {pooled.mean().item():.4f}")
        print(f"   Std: {pooled.std().item():.4f}")

    # 3. Hidden states (all layers)
    if hasattr(outputs, 'hidden_states') and outputs.hidden_states is not None:
        print(f"\n3. hidden_states (all layers):")
        print(f"   Num layers: {len(outputs.hidden_states)}")
        for idx, hs in enumerate(outputs.hidden_states[-3:]):  # Last 3 layers
            print(f"   Layer {len(outputs.hidden_states)-3+idx}: shape={hs.shape}, "
                  f"mean={hs.mean().item():.4f}, std={hs.std().item():.4f}")

    print("="*60 + "\n")

# Run debug
debug_outputs()

# ========================== EMBEDDING - TRY MULTIPLE METHODS ==========================
@torch.inference_mode()
def batch_embed_method1(crops, batch_size=8):
    """Method 1: pooler_output WITHOUT normalization (test)"""
    if len(crops) == 0:
        return torch.empty((0, 640), device="cuda")

    all_embs = []
    for i in range(0, len(crops), batch_size):
        batch_crops = crops[i:i+batch_size]
        inputs = processor(images=batch_crops, return_tensors="pt").to("cuda")
        outputs = convnext(**inputs)

        embs = outputs.pooler_output  # NO normalization
        all_embs.append(embs.cpu())
        del inputs, outputs

    result = torch.cat(all_embs, dim=0).cuda()
    del all_embs
    torch.cuda.empty_cache()
    return result

@torch.inference_mode()
def batch_embed_method2(crops, batch_size=8):
    """Method 2: Manual pooling from last_hidden_state"""
    if len(crops) == 0:
        return torch.empty((0, 640), device="cuda")

    all_embs = []
    for i in range(0, len(crops), batch_size):
        batch_crops = crops[i:i+batch_size]
        inputs = processor(images=batch_crops, return_tensors="pt").to("cuda")
        outputs = convnext(**inputs)

        # Manual global average pooling
        features = outputs.last_hidden_state  # [B, 640, 7, 7]
        pooled = features.mean(dim=[2, 3])  # [B, 640]

        # NO normalization first
        all_embs.append(pooled.cpu())
        del inputs, outputs, features

    result = torch.cat(all_embs, dim=0).cuda()
    del all_embs
    torch.cuda.empty_cache()
    return result

@torch.inference_mode()
def batch_embed_method3(crops, batch_size=8):
    """Method 3: Max pooling instead of mean"""
    if len(crops) == 0:
        return torch.empty((0, 640), device="cuda")

    all_embs = []
    for i in range(0, len(crops), batch_size):
        batch_crops = crops[i:i+batch_size]
        inputs = processor(images=batch_crops, return_tensors="pt").to("cuda")
        outputs = convnext(**inputs)

        # Max pooling
        features = outputs.last_hidden_state  # [B, 640, 7, 7]
        pooled = torch.nn.functional.adaptive_max_pool2d(features, (1, 1)).squeeze(-1).squeeze(-1)

        all_embs.append(pooled.cpu())
        del inputs, outputs, features

    result = torch.cat(all_embs, dim=0).cuda()
    del all_embs
    torch.cuda.empty_cache()
    return result

@torch.inference_mode()
def batch_embed_method4(crops, batch_size=8):
    """Method 4: Flatten + PCA-like reduction"""
    if len(crops) == 0:
        return torch.empty((0, 640), device="cuda")

    all_embs = []
    for i in range(0, len(crops), batch_size):
        batch_crops = crops[i:i+batch_size]
        inputs = processor(images=batch_crops, return_tensors="pt").to("cuda")
        outputs = convnext(**inputs)

        # Flatten spatial dims and take mean
        features = outputs.last_hidden_state  # [B, 640, 7, 7]
        B, C, H, W = features.shape
        features_flat = features.reshape(B, C, -1)  # [B, 640, 49]

        # Weight by magnitude (similar to attention)
        weights = features_flat.abs().sum(dim=1, keepdim=True)  # [B, 1, 49]
        weights = weights / (weights.sum(dim=2, keepdim=True) + 1e-8)

        pooled = (features_flat * weights).sum(dim=2)  # [B, 640]

        all_embs.append(pooled.cpu())
        del inputs, outputs, features

    result = torch.cat(all_embs, dim=0).cuda()
    del all_embs
    torch.cuda.empty_cache()
    return result

# ========================== TEST ALL METHODS ==========================
def get_template_test(ref_paths):
    """Test all methods"""
    if len(ref_paths) == 0:
        raise ValueError("❌ No reference images found!")

    imgs = []
    for p in ref_paths:
        img = cv2.imread(str(p))
        if img is None:
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        imgs.append(img_rgb)

    if len(imgs) == 0:
        raise ValueError("❌ No images loaded!")

    print(f"\n🧪 TESTING ALL EMBEDDING METHODS")
    print("="*60)

    results = {}

    for method_name, method_func in [
        ("Method 1 (pooler_output, no norm)", batch_embed_method1),
        ("Method 2 (mean pooling)", batch_embed_method2),
        ("Method 3 (max pooling)", batch_embed_method3),
        ("Method 4 (weighted pooling)", batch_embed_method4),
    ]:
        print(f"\nTesting: {method_name}")
        embs = method_func(imgs)

        print(f"  Shape: {embs.shape}")
        print(f"  Mean: {embs.mean().item():.4f}")
        print(f"  Std: {embs.std().item():.4f}")
        print(f"  Range: [{embs.min().item():.4f}, {embs.max().item():.4f}]")

        # Create template
        template = embs.mean(dim=0)

        # Test self-similarity (should be ~1.0)
        sims = torch.nn.functional.cosine_similarity(embs, template.unsqueeze(0), dim=1)
        print(f"  Self-similarity: {sims.mean().item():.4f} ± {sims.std().item():.4f}")
        print(f"  Self-sim range: [{sims.min().item():.4f}, {sims.max().item():.4f}]")

        results[method_name] = {
            'template': template,
            'self_sim': sims.mean().item()
        }

    print("\n" + "="*60)
    print("RECOMMENDATION:")
    best_method = max(results.items(), key=lambda x: x[1]['self_sim'])
    print(f"✅ Best method: {best_method[0]}")
    print(f"   Self-similarity: {best_method[1]['self_sim']:.4f}")

    return best_method[1]['template'], best_method[0]

# ========================== MAIN ==========================
ref_folder = Path("/content/drive/MyDrive/ZaloAI/public_test/samples/BlackBox_0/object_images")

if not ref_folder.exists():
    raise FileNotFoundError(f"Folder not found: {ref_folder}")

ref_paths = sorted(ref_folder.glob("*.jpg"))
if len(ref_paths) == 0:
    ref_paths = sorted(ref_folder.glob("*.png"))

print(f"Found {len(ref_paths)} reference images")

if len(ref_paths) == 0:
    raise ValueError("No reference images!")

# Test and get best template
template, best_method_name = get_template_test(ref_paths)

print(f"\n✅ Using: {best_method_name}")
print(f"Template shape: {template.shape}")

✅ Loaded ConvNeXtV2-Nano: 15M params


RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

In [ ]:
import cv2
import numpy as np
from transformers import AutoImageProcessor, ConvNextV2Model
import torch
from pathlib import Path
from ultralytics.models.fastsam import FastSAMPredictor

# Load models
overrides = dict(
    conf=0.25,
    task="segment",
    mode="predict",
    model="FastSAM-s.pt",
    imgsz=640,
    retina_masks=False
)
sam = FastSAMPredictor(overrides=overrides)

processor = AutoImageProcessor.from_pretrained("facebook/convnextv2-tiny-22k-224")
convnext = ConvNextV2Model.from_pretrained("facebook/convnextv2-tiny-22k-224").cuda().eval()

print("✅ Loaded ConvNeXtV2-Tiny")

# ========================== EMBEDDING - MAX POOLING ==========================
@torch.inference_mode()
def batch_embed(crops, batch_size=8):
    """ConvNeXt-V2 embeddings - MAX POOLING (BEST METHOD)"""
    if len(crops) == 0:
        return torch.empty((0, 640), device="cuda")

    all_embs = []

    for i in range(0, len(crops), batch_size):
        batch_crops = crops[i:i+batch_size]

        inputs = processor(images=batch_crops, return_tensors="pt").to("cuda")
        outputs = convnext(**inputs)

        # ✅ MAX POOLING - gives best self-similarity
        features = outputs.last_hidden_state  # [B, 640, 7, 7]
        pooled = torch.nn.functional.adaptive_max_pool2d(features, (1, 1))
        pooled = pooled.squeeze(-1).squeeze(-1)  # [B, 640]

        all_embs.append(pooled.cpu())
        del inputs, outputs, features
        torch.cuda.empty_cache()

    result = torch.cat(all_embs, dim=0).cuda()
    del all_embs
    torch.cuda.empty_cache()

    return result

#========================== GET TEMPLATE ==========================
def get_template(ref_paths):
    if len(ref_paths) == 0:
        raise ValueError("❌ No reference images found!")

    imgs = []
    print(f"\n🔍 Loading {len(ref_paths)} reference images...")

    for p in ref_paths:
        img = cv2.imread(str(p))
        if img is None:
            print(f"⚠️ Failed to read {p}")
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        imgs.append(img_rgb)
        print(f"  ✅ Loaded {p.name}")

    if len(imgs) == 0:
        raise ValueError("❌ No images loaded!")

    print(f"\n🧠 Embedding {len(imgs)} images with MAX POOLING...")
    embs = batch_embed(imgs)
    print(f"✅ Embeddings shape: {embs.shape}")
    print(f"✅ Embeddings range: [{embs.min():.4f}, {embs.max():.4f}]")

    # Create template (mean of embeddings)
    template = embs.mean(dim=0)

    # Test self-similarity
    sims = torch.nn.functional.cosine_similarity(embs, template.unsqueeze(0), dim=1)
    print(f"✅ Self-similarity: {sims.mean().item():.4f} (should be ~0.95)")

    return template

# ========================== MAIN ==========================
ref_folder = Path("/content/drive/MyDrive/ZaloAI/public_test/samples/BlackBox_0/object_images")

if not ref_folder.exists():
    print(f"❌ Folder not found: {ref_folder}")
    raise FileNotFoundError()

ref_paths = sorted(ref_folder.glob("*.jpg"))
if len(ref_paths) == 0:
    ref_paths = sorted(ref_folder.glob("*.png"))

print(f"Found {len(ref_paths)} reference images")

if len(ref_paths) == 0:
    raise ValueError("No reference images!")

template = get_template(ref_paths)

# Load drone frame
frame = cv2.imread("drone-blackbox-14.png")
if frame is None:
    raise ValueError("Failed to load frame!")

img_h, img_w = frame.shape[:2]
print(f"\n{'='*60}")
print(f"Frame gốc: {img_w}x{img_h}")
print(f"{'='*60}")

results = sam(frame)[0]

if results.masks is not None and results.boxes is not None:
    masks = results.masks.data.cpu().numpy()
    boxes_xyxy = results.boxes.xyxy.cpu().numpy()
    confidences = results.boxes.conf.cpu().numpy() if results.boxes.conf is not None else None

    print(f"Tổng số masks SAM tạo ra: {len(masks)}")

    # BƯỚC 1: Lọc masks theo pixels
    valid_mask_indices = []
    valid_areas = []
    valid_confidences = []

    min_pixels = 10
    max_pixels = 3000

    for idx, mask in enumerate(masks):
        pos = np.where(mask)
        num_pixels = len(pos[0])

        if num_pixels < min_pixels or num_pixels > max_pixels:
            continue

        valid_mask_indices.append(idx)
        valid_areas.append(num_pixels)
        if confidences is not None and idx < len(confidences):
            valid_confidences.append(confidences[idx])

    print(f"Sau khi lọc {min_pixels}-{max_pixels} pixels: {len(valid_mask_indices)} masks")

    # BƯỚC 2: Lấy top 25
    if len(valid_mask_indices) > 25:
        if valid_confidences and len(valid_confidences) == len(valid_mask_indices):
            topk_idx = np.argsort(valid_confidences)[::-1][:25]
            print(f"✅ Lấy top 25 theo CONFIDENCE")
        else:
            topk_idx = np.argsort(valid_areas)[::-1][:25]
            print(f"⚠️ Lấy top 25 theo AREA")

        valid_mask_indices = [valid_mask_indices[i] for i in topk_idx]
        valid_areas = [valid_areas[i] for i in topk_idx]
        if valid_confidences:
            valid_confidences = [valid_confidences[i] for i in topk_idx]

    print(f"Sau khi lấy top 25: {len(valid_mask_indices)} masks")

    # BƯỚC 2.5: Lọc bbox quá lớn
    max_bbox_width = img_w * 0.1
    max_bbox_height = img_h * 0.1
    max_bbox_area = (img_w * img_h) * 0.01

    size_filtered_indices = []
    size_filtered_areas = []
    size_filtered_confidences = []

    for i, idx in enumerate(valid_mask_indices):
        bbox = boxes_xyxy[idx]
        x1, y1, x2, y2 = bbox

        bbox_w = x2 - x1
        bbox_h = y2 - y1
        bbox_area = bbox_w * bbox_h

        if bbox_w > max_bbox_width or bbox_h > max_bbox_height or bbox_area > max_bbox_area:
            continue

        size_filtered_indices.append(idx)
        size_filtered_areas.append(valid_areas[i])
        if valid_confidences and i < len(valid_confidences):
            size_filtered_confidences.append(valid_confidences[i])

    print(f"Sau khi lọc kích thước bbox: {len(size_filtered_indices)} masks")

    # BƯỚC 3: Crop và embed
    crops_rgb = []
    valid_bboxes = []
    final_areas = []
    final_confidences = []

    for i, idx in enumerate(size_filtered_indices):
        bbox = boxes_xyxy[idx]
        x1, y1, x2, y2 = bbox

        x1 = max(0, int(x1))
        y1 = max(0, int(y1))
        x2 = min(img_w - 1, int(x2))
        y2 = min(img_h - 1, int(y2))

        if x1 >= x2 or y1 >= y2:
            continue

        crop = frame[y1:y2+1, x1:x2+1]
        if crop.size == 0 or crop.shape[0] == 0 or crop.shape[1] == 0:
            continue

        try:
            crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

            crops_rgb.append(crop_rgb)
            valid_bboxes.append([x1, y1, x2, y2])
            final_areas.append(size_filtered_areas[i])
            if size_filtered_confidences and i < len(size_filtered_confidences):
                final_confidences.append(size_filtered_confidences[i])
        except Exception as e:
            continue

    print(f"Số crops để so khớp: {len(crops_rgb)}")

    # BƯỚC 4: So khớp với template
    if len(crops_rgb) > 0:
        embs = batch_embed(crops_rgb)
        sims = torch.nn.functional.cosine_similarity(embs, template.unsqueeze(0), dim=1)

        print(f"\n{'='*60}")
        print("Similarity scores (with MAX POOLING):")
        for i, (sim, bbox) in enumerate(zip(sims, valid_bboxes)):
            x1, y1, x2, y2 = bbox
            bbox_w = x2 - x1
            bbox_h = y2 - y1
            conf_str = f", conf={final_confidences[i]:.3f}" if final_confidences and i < len(final_confidences) else ""

            print(f"  Mask {i}: sim={sim.item():.4f}{conf_str}, bbox=[{x1},{y1},{x2},{y2}], "
                  f"w={bbox_w}, h={bbox_h}")

        # ✅ Threshold phù hợp với max pooling
        threshold = 0.8
        filtered_indices = []
        filtered_sims = []
        filtered_bboxes = []

        for i, (sim, bbox) in enumerate(zip(sims, valid_bboxes)):
            if sim.item() > threshold:
                filtered_indices.append(i)
                filtered_sims.append(sim.item())
                filtered_bboxes.append(bbox)

        print(f"\n{'='*60}")
        print(f"Sau khi lọc sim > {threshold}: {len(filtered_bboxes)} masks")

        best_original_idx = None
        best_sim = 0.0
        best_bbox = None

        if len(filtered_bboxes) > 0:
            best_filtered_idx = np.argmax(filtered_sims)
            best_original_idx = filtered_indices[best_filtered_idx]
            best_sim = filtered_sims[best_filtered_idx]
            best_bbox = filtered_bboxes[best_filtered_idx]

            print(f"\n*** BEST MATCH ***")
            print(f"  Mask index: {best_original_idx}")
            print(f"  Similarity: {best_sim:.4f}")
            print(f"  BBox: {best_bbox}")
            if final_confidences and best_original_idx < len(final_confidences):
                print(f"  Confidence: {final_confidences[best_original_idx]:.4f}")
            print(f"✅ Object DETECTED")
        else:
            print(f"\n✗ Không có mask nào có sim > {threshold}")
            print(f"  Best sim: {sims.max().item():.4f}")

        # Visualize
        frame_vis = frame.copy()

        for i, bbox in enumerate(valid_bboxes):
            x1, y1, x2, y2 = [int(v) for v in bbox]

            if best_original_idx is not None and i == best_original_idx:
                color = (0, 0, 255)
                thickness = 3
                sim_val = best_sim
                label = f"BEST: {sim_val:.2f}"
            else:
                color = (0, 255, 0)
                thickness = 1
                sim_val = sims[i].item()
                label = f"{sim_val:.2f}"

            cv2.rectangle(frame_vis, (x1, y1), (x2, y2), color, thickness)
            text_y = max(y1 - 5, 15)
            cv2.putText(frame_vis, label, (x1, text_y),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        cv2.imwrite("test_best_match.png", frame_vis)
        print(f"\n✅ Đã lưu: test_best_match.png")
    else:
        print("Không có crops nào để so khớp!")
else:
    print("Không có masks hoặc boxes!")

✅ Loaded ConvNeXtV2-Tiny
Found 3 reference images

🔍 Loading 3 reference images...
  ✅ Loaded img_1.jpg
  ✅ Loaded img_2.jpg
  ✅ Loaded img_3.jpg

🧠 Embedding 3 images with MAX POOLING...
✅ Embeddings shape: torch.Size([3, 768])
✅ Embeddings range: [-2.4479, 29.5361]
✅ Self-similarity: 0.9554 (should be ~0.95)

Frame gốc: 1024x576

Ultralytics 8.3.230 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8s-seg summary (fused): 85 layers, 11,779,987 parameters, 0 gradients, 39.9 GFLOPs
0: 640x640 17 objects, 20.8ms
Speed: 2.2ms preprocess, 20.8ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/segment/predict4
Tổng số masks SAM tạo ra: 17
Sau khi lọc 10-3000 pixels: 14 masks
Sau khi lấy top 25: 14 masks
Sau khi lọc kích thước bbox: 6 masks
Số crops để so khớp: 6

Similarity scores (with MAX POOLING):
  Mask 0: sim=0.8058, conf=0.848, bbox=[911,321,968,371], w=57, h=50
  Mask 1: sim=0.8047, conf=0.784, bbox=[214,212,246,236],

In [ ]:
# ========================== IMPORTS ==========================
import os
import gc
import cv2
import torch
import numpy as np
from pathlib import Path
import json
from tqdm import tqdm
from ultralytics.models.fastsam import FastSAMPredictor
from ultralytics import SAM
from transformers import AutoImageProcessor, ConvNextV2Model


# ========================== LOAD MODELS ==========================
print("Đang load FastSAM + ConvNeXtV2-Nano...")

# FastSAM Predictor
overrides = dict(
    conf=0.25,
    task="segment",
    mode="predict",
    model="FastSAM-s.pt",
    imgsz=640,
    retina_masks=False,
    max_det=50
)

sam_template = SAM("mobile_sam.pt")
sam = FastSAMPredictor(overrides=overrides)

processor = AutoImageProcessor.from_pretrained("facebook/convnextv2-tiny-22k-224")
convnext = ConvNextV2Model.from_pretrained("facebook/convnextv2-tiny-22k-224").cuda().eval()

print("✅ Loaded ConvNeXtV2-Nano: 15M params")

# ========================== EMBEDDING - MAX POOLING ==========================
@torch.inference_mode()
def batch_embed(crops, batch_size=8):
    """ConvNeXt-V2 embeddings - MAX POOLING (BEST METHOD)"""
    if len(crops) == 0:
        return torch.empty((0, 640), device="cuda")

    all_embs = []

    for i in range(0, len(crops), batch_size):
        batch_crops = crops[i:i+batch_size]

        inputs = processor(images=batch_crops, return_tensors="pt").to("cuda")
        outputs = convnext(**inputs)

        # ✅ MAX POOLING - gives best self-similarity
        features = outputs.last_hidden_state  # [B, 640, 7, 7]
        pooled = torch.nn.functional.adaptive_max_pool2d(features, (1, 1))
        pooled = pooled.squeeze(-1).squeeze(-1)  # [B, 640]

        all_embs.append(pooled.cpu())
        del inputs, outputs, features
        torch.cuda.empty_cache()

    result = torch.cat(all_embs, dim=0).cuda()
    del all_embs
    torch.cuda.empty_cache()

    return result

# ========================== REFERENCE TEMPLATE ==========================
def get_template(ref_paths, sam_model, video_id):
    print(f"\n🔍 Processing {len(ref_paths)} reference images...")
    imgs = []

    template_dir = Path(f"templates/{video_id}")
    template_dir.mkdir(parents=True, exist_ok=True)
    print(f"📁 Saving template crops to: {template_dir}")

    for idx, p in enumerate(ref_paths):
        print(f"  📸 Image {idx+1}/{len(ref_paths)}: {p.name}")

        img_bgr = cv2.imread(str(p))
        if img_bgr is None:
            print(f"    ⚠️  Failed to read image")
            continue

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        print(f"    📐 Original size: {img_rgb.shape[1]}x{img_rgb.shape[0]}")

        results = sam_model(img_rgb)
        result = results[0] if len(results) > 0 else None

        # ✅ GIỐNG CODE BẠN - Đơn giản hơn
        if result is None or result.masks is None or result.boxes is None or result.boxes.conf is None:
            print(f"    ⚠️  No masks/boxes/scores found, using full image")
            crop = img_rgb
        else:
            masks = result.masks.data.cpu().numpy()
            scores = result.boxes.conf.detach().cpu().numpy()

            # ✅ Lấy best mask theo score
            best_idx = int(np.argmax(scores))
            best_conf = float(scores[best_idx])

            print(f"    ⭐ Best mask: #{best_idx} (score: {best_conf:.3f})")

            if best_idx >= len(masks):
                print(f"    ⚠️  Index mismatch, using full image")
                crop = img_rgb
            else:
                best_mask = masks[best_idx]
                ys, xs = np.where(best_mask)

                if ys.size == 0 or xs.size == 0:
                    print(f"    ⚠️  Empty mask, using full image")
                    crop = img_rgb
                else:
                    # ✅ GIỐNG CODE BẠN - Không padding
                    y1, y2 = ys.min(), ys.max()
                    x1, x2 = xs.min(), xs.max()
                    area = ys.size

                    print(f"    ✂️  Cropped to: bbox=[{x1},{y1},{x2},{y2}], area={area} pixels")

                    crop = img_rgb[y1:y2+1, x1:x2+1]

                    if crop.size == 0:
                        print(f"    ⚠️  Empty crop, using full image")
                        crop = img_rgb

        del results, result
        torch.cuda.empty_cache()

        # Lưu crop
        crop_bgr = cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)
        crop_filename = template_dir / f"crop_{idx+1}_{p.stem}.jpg"
        cv2.imwrite(str(crop_filename), crop_bgr)
        print(f"    💾 Saved crop to: {crop_filename.name}")

        imgs.append(crop)
        print(f"    ✅ Added crop")

        del img_bgr, img_rgb, crop, crop_bgr
        torch.cuda.empty_cache()

    if len(imgs) == 0:
        raise ValueError("❌ Không tạo được crop nào từ ảnh tham chiếu.")

    print(f"\n🧠 Embedding {len(imgs)} images with MAX POOLING...")
    embs = batch_embed(imgs)
    print(f"✅ Embeddings shape: {embs.shape}")
    print(f"✅ Embeddings range: [{embs.min():.4f}, {embs.max():.4f}]")

    template = embs.mean(dim=0)

    sims = torch.nn.functional.cosine_similarity(embs, template.unsqueeze(0), dim=1)
    print(f"✅ Self-similarity: {sims.mean().item():.4f} (should be ~0.95)")

    print(f"✅ Template created! Shape: {template.shape}")
    print(f"✅ Template crops saved to: {template_dir}/")

    del embs, sims
    torch.cuda.empty_cache()

    return template

# ========================== HELPER FUNCTIONS ==========================
def bbox_center(bbox):
    return ((bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2)

def bbox_distance(bbox1, bbox2):
    c1 = bbox_center(bbox1)
    c2 = bbox_center(bbox2)
    return np.sqrt((c1[0] - c2[0])**2 + (c1[1] - c2[1])**2)

def bbox_iou(bbox1, bbox2):
    x1 = max(bbox1[0], bbox2[0])
    y1 = max(bbox1[1], bbox2[1])
    x2 = min(bbox1[2], bbox2[2])
    y2 = min(bbox1[3], bbox2[3])

    if x2 <= x1 or y2 <= y1:
        return 0.0

    inter = (x2 - x1) * (y2 - y1)
    area1 = (bbox1[2] - bbox1[0]) * (bbox1[3] - bbox1[1])
    area2 = (bbox2[2] - bbox2[0]) * (bbox2[3] - bbox2[1])
    union = area1 + area2 - inter

    return inter / union if union > 0 else 0.0

# ========================== MAIN PROCESS ==========================
@torch.inference_mode()
def process_video(folder: Path):
    video_path = folder / "drone_video_test_1.mp4"
    refs = sorted((folder / "object_images").glob("*.jpg"))
    video_id = folder.name

    print(f"\n{'='*60}")
    print(f"📹 Video: {video_id}")
    print(f"{'='*60}")

    template = get_template(refs, sam_template, video_id)

    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"\n🎬 Processing video: {total_frames} frames")

    detections = []
    skip_counter = 0

    last_bbox = None
    frames_since_last_det = 0
    max_frames_lost = 30

    pbar = tqdm(total=total_frames, desc=video_id, leave=False)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx = int(pbar.n)
        img_h, img_w = frame.shape[:2]

        should_detect = (skip_counter % 2 == 0)

        if should_detect:
            torch.cuda.empty_cache()

            results = sam(frame)[0]

            # ✅ Y HỆT CODE TEST - Kiểm tra masks và boxes
            if results.masks is None or results.boxes is None:
                frames_since_last_det += 1
                if frames_since_last_det > max_frames_lost:
                    last_bbox = None

                del results
                torch.cuda.empty_cache()

                pbar.update(1)
                skip_counter += 1
                continue

            # ✅ Y HỆT - Lấy boxes và masks
            masks = results.masks.data.cpu().numpy()
            boxes_xyxy = results.boxes.xyxy.cpu().numpy()
            confidences = results.boxes.conf.cpu().numpy() if results.boxes.conf is not None else None

            del results
            torch.cuda.empty_cache()

            # ✅ Y HỆT - BƯỚC 1: Lọc masks theo pixels
            valid_mask_indices = []
            valid_areas = []
            valid_confidences = []

            min_pixels = 10
            max_pixels = 3000

            for idx, mask in enumerate(masks):
                pos = np.where(mask)
                num_pixels = len(pos[0])

                if num_pixels < min_pixels or num_pixels > max_pixels:
                    continue

                valid_mask_indices.append(idx)
                valid_areas.append(num_pixels)
                if confidences is not None and idx < len(confidences):
                    valid_confidences.append(confidences[idx])

            # ✅ Y HỆT - BƯỚC 2: Lấy top 25
            if len(valid_mask_indices) > 25:
                if valid_confidences and len(valid_confidences) == len(valid_mask_indices):
                    topk_idx = np.argsort(valid_confidences)[::-1][:25]
                else:
                    topk_idx = np.argsort(valid_areas)[::-1][:25]

                valid_mask_indices = [valid_mask_indices[i] for i in topk_idx]
                valid_areas = [valid_areas[i] for i in topk_idx]
                if valid_confidences:
                    valid_confidences = [valid_confidences[i] for i in topk_idx]

            del masks, confidences

            # ✅ Y HỆT - BƯỚC 2.5: Lọc bbox quá lớn
            max_bbox_width = img_w * 0.1
            max_bbox_height = img_h * 0.1
            max_bbox_area = (img_w * img_h) * 0.01

            size_filtered_indices = []
            size_filtered_areas = []
            size_filtered_confidences = []

            for i, idx in enumerate(valid_mask_indices):
                bbox = boxes_xyxy[idx]
                x1, y1, x2, y2 = bbox

                bbox_w = x2 - x1
                bbox_h = y2 - y1
                bbox_area = bbox_w * bbox_h

                if bbox_w > max_bbox_width or bbox_h > max_bbox_height or bbox_area > max_bbox_area:
                    continue

                size_filtered_indices.append(idx)
                size_filtered_areas.append(valid_areas[i])
                if valid_confidences and i < len(valid_confidences):
                    size_filtered_confidences.append(valid_confidences[i])

            del valid_mask_indices, valid_areas, valid_confidences

            # ✅ Y HỆT - BƯỚC 3: Crop và embed
            crops_rgb = []
            valid_bboxes = []
            final_areas = []
            final_confidences = []

            for i, idx in enumerate(size_filtered_indices):
                bbox = boxes_xyxy[idx]
                x1, y1, x2, y2 = bbox

                x1 = max(0, int(x1))
                y1 = max(0, int(y1))
                x2 = min(img_w - 1, int(x2))
                y2 = min(img_h - 1, int(y2))

                if x1 >= x2 or y1 >= y2:
                    continue

                crop = frame[y1:y2+1, x1:x2+1]
                if crop.size == 0 or crop.shape[0] == 0 or crop.shape[1] == 0:
                    continue

                try:
                    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

                    crops_rgb.append(crop_rgb)
                    valid_bboxes.append([x1, y1, x2, y2])
                    final_areas.append(size_filtered_areas[i])
                    if size_filtered_confidences and i < len(size_filtered_confidences):
                        final_confidences.append(size_filtered_confidences[i])
                except Exception as e:
                    continue

            del size_filtered_indices, size_filtered_areas, size_filtered_confidences

            # ✅ Y HỆT - BƯỚC 4: So khớp với template
            if crops_rgb:
                embs = batch_embed(crops_rgb)
                sims = torch.nn.functional.cosine_similarity(embs, template.unsqueeze(0), dim=1)

                sims_cpu = sims.cpu().numpy()
                del embs, sims
                torch.cuda.empty_cache()

                # ✅ Threshold 0.8 như code test
                sim_threshold = 0.82
                candidates = []
                for idx in range(len(sims_cpu)):
                    if sims_cpu[idx] > sim_threshold:
                        candidates.append({
                            'idx': idx,
                            'sim': float(sims_cpu[idx]),
                            'bbox': valid_bboxes[idx]
                        })

                del sims_cpu, crops_rgb, valid_bboxes

                if candidates:
                    best_candidate = None

                    if last_bbox is None:
                        best_candidate = max(candidates, key=lambda x: x['sim'])
                    else:
                        max_velocity = 50 * (frames_since_last_det + 1)
                        max_distance = min(max_velocity, 200)

                        nearby_candidates = []
                        for cand in candidates:
                            dist = bbox_distance(last_bbox, cand['bbox'])
                            if dist <= max_distance:
                                proximity_score = 1 - (dist / max_distance)
                                combined_score = 0.6 * cand['sim'] + 0.4 * proximity_score
                                cand['combined_score'] = combined_score
                                cand['distance'] = dist
                                nearby_candidates.append(cand)

                        if nearby_candidates:
                            best_candidate = max(nearby_candidates, key=lambda x: x['combined_score'])
                        else:
                            high_sim_candidates = [c for c in candidates if c['sim'] > 0.85]
                            if high_sim_candidates:
                                best_candidate = max(high_sim_candidates, key=lambda x: x['sim'])
                                last_bbox = None

                    if best_candidate:
                        x1, y1, x2, y2 = best_candidate['bbox']
                        detections.append({
                            "frame": frame_idx,
                            "x1": int(x1),
                            "y1": int(y1),
                            "x2": int(x2),
                            "y2": int(y2)
                        })
                        last_bbox = best_candidate['bbox']
                        frames_since_last_det = 0
                    else:
                        frames_since_last_det += 1
                else:
                    frames_since_last_det += 1
            else:
                frames_since_last_det += 1

            if frames_since_last_det > max_frames_lost:
                last_bbox = None

        skip_counter += 1
        pbar.update(1)

        if frame_idx % 30 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    cap.release()
    pbar.close()

    print(f"✅ Detected {len(detections)} bboxes before interpolation")

    gc.collect()
    torch.cuda.empty_cache()

    # INTERPOLATION
    if len(detections) > 1:
        full_dets = []
        max_gap = 20
        max_interp_distance = 100

        sorted_dets = sorted(detections, key=lambda x: x["frame"])

        for i in range(len(sorted_dets)):
            det = sorted_dets[i]
            full_dets.append(det)

            if i < len(sorted_dets) - 1:
                next_det = sorted_dets[i + 1]
                gap = next_det["frame"] - det["frame"]

                bbox1 = [det["x1"], det["y1"], det["x2"], det["y2"]]
                bbox2 = [next_det["x1"], next_det["y1"], next_det["x2"], next_det["y2"]]
                distance = bbox_distance(bbox1, bbox2)

                if 1 < gap <= max_gap and distance <= max_interp_distance:
                    for f in range(det["frame"] + 1, next_det["frame"]):
                        alpha = (f - det["frame"]) / gap
                        bbox = {
                            "frame": f,
                            "x1": int(det["x1"] * (1-alpha) + next_det["x1"] * alpha),
                            "y1": int(det["y1"] * (1-alpha) + next_det["y1"] * alpha),
                            "x2": int(det["x2"] * (1-alpha) + next_det["x2"] * alpha),
                            "y2": int(det["y2"] * (1-alpha) + next_det["y2"] * alpha),
                        }
                        full_dets.append(bbox)

        detections = sorted(full_dets, key=lambda x: x["frame"])
        print(f"✅ After interpolation: {len(detections)} bboxes")

    if len(detections) > 0:
        final_output_detections = [{"bboxes": detections}]
    else:
        final_output_detections = []

    return {"video_id": video_id, "detections": final_output_detections}

# ========================== MAIN ==========================
if __name__ == "__main__":
    DATA_DIR = "/content/drive/MyDrive/ZaloAI/public_test/samples"
    OUTPUT_FILE = "/content/submission.json"

    results = []
    data_path = Path(DATA_DIR)

    if not data_path.exists():
        print(f"❌ Lỗi: Không tìm thấy thư mục {DATA_DIR}")
    else:
        folders = sorted([f for f in data_path.iterdir() if (f / "drone_video_test_1.mp4").exists()])
        print(f"\n🎯 Tìm thấy {len(folders)} video để xử lý\n")

        for folder in folders:
            try:
                res = process_video(folder)
                results.append(res)
            except Exception as e:
                print(f"❌ Error processing {folder.name}: {e}")
                import traceback
                traceback.print_exc()
                results.append({"video_id": folder.name, "detections": []})

            gc.collect()
            torch.cuda.empty_cache()

        json.dump(results, open(OUTPUT_FILE, "w"), indent=2)
        print(f"\n✅ Hoàn tất! Output: {OUTPUT_FILE}")

        import shutil
        drive_output = f"{DATA_DIR}/submission.json"
        shutil.copy(OUTPUT_FILE, drive_output)
        print(f"✅ Đã copy kết quả vào: {drive_output}")

Đang load FastSAM + ConvNeXtV2-Nano...
✅ Loaded ConvNeXtV2-Nano: 15M params

🎯 Tìm thấy 1 video để xử lý


📹 Video: BlackBox_0

🔍 Processing 3 reference images...
📁 Saving template crops to: templates/BlackBox_0
  📸 Image 1/3: img_1.jpg
    📐 Original size: 2992x2992

0: 1024x1024 1 0, 1 1, 1 2, 1 3, 1 4, 1 5, 1 6, 1 7, 1 8, 1 9, 1 10, 1 11, 1 12, 1 13, 1 14, 1 15, 1 16, 1 17, 1 18, 1 19, 1 20, 1 21, 1 22, 1 23, 1 24, 1 25, 1 26, 1 27, 1 28, 1 29, 1 30, 1 31, 1 32, 1 33, 1 34, 1 35, 1 36, 1 37, 1 38, 1 39, 1 40, 1 41, 1 42, 1 43, 1 44, 1 45, 1 46, 1 47, 1 48, 1 49, 1 50, 1 51, 1 52, 1 53, 1 54, 1 55, 1 56, 1 57, 1 58, 1 59, 1 60, 1 61, 4741.8ms
Speed: 19.1ms preprocess, 4741.8ms inference, 30.0ms postprocess per image at shape (1, 3, 1024, 1024)
    ⭐ Best mask: #0 (score: 1.022)
    ✂️  Cropped to: bbox=[355,1012,2773,2440], area=2854246 pixels
    💾 Saved crop to: crop_1_img_1.jpg
    ✅ Added crop
  📸 Image 2/3: img_2.jpg
    📐 Original size: 2992x2992

0: 1024x1024 1 0, 1 1, 1 2, 1 


BlackBox_0:   0%|          | 0/440 [00:00<?, ?it/s]


Ultralytics 8.3.230 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8s-seg summary (fused): 85 layers, 11,779,987 parameters, 0 gradients, 39.9 GFLOPs
0: 640x640 1 object, 9.8ms
Speed: 3.7ms preprocess, 9.8ms inference, 2.9ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/segment/predict60



BlackBox_0:   0%|          | 2/440 [00:00<03:19,  2.20it/s]


0: 640x640 1 object, 12.7ms
Speed: 6.0ms preprocess, 12.7ms inference, 3.8ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/segment/predict60

0: 640x640 1 object, 9.7ms
Speed: 3.6ms preprocess, 9.7ms inference, 2.3ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/segment/predict60


...


BlackBox_0:  98%|█████████▊| 433/440 [00:29<00:00, 15.16it/s]


0: 640x640 1 object, 19.5ms
Speed: 3.4ms preprocess, 19.5ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/segment/predict60

0: 640x640 1 object, 19.2ms
Speed: 3.5ms preprocess, 19.2ms inference, 2.8ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/segment/predict60



BlackBox_0:  99%|█████████▉| 437/440 [00:29<00:00, 18.31it/s]


0: 640x640 1 object, 19.2ms
Speed: 3.3ms preprocess, 19.2ms inference, 3.3ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/segment/predict60


✅ Detected 139 bboxes before interpolation
✅ After interpolation: 357 bboxes

✅ Hoàn tất! Output: /content/submission.json
✅ Đã copy kết quả vào: /content/drive/MyDrive/ZaloAI/public_test/samples/submission.json
